# LLM QualityGuard

## Automated LLM Evaluation & Data Quality Platform

LLM QualityGuard is a data and AI evaluation platform designed to measure the quality, reliability, and grounding of Large Language Model (LLM) responses.

The project combines:

* LLM evaluation
* LLM-as-a-judge
* semantic embeddings
* Retrieval-Augmented Generation (RAG)
* retrieval evaluation
* hallucination detection
* SQL analytics
* automated testing
* workflow orchestration
* Power BI reporting

The goal is to move from simply asking an LLM a question to systematically measuring whether its answer is reliable and supported by trusted information.


# 2. Project Objectives

Large Language Models can generate answers that sound convincing even when the information is incorrect, incomplete, or unsupported.

LLM QualityGuard is designed to answer four key questions:

1. **How accurate is the LLM's answer?**
2. **Can the correct company policy be retrieved for the question?**
3. **Is the generated answer grounded in the retrieved policy?**
4. **Can the evaluation results be stored, analysed, tested, and monitored?**

The project therefore evaluates two connected areas:

### LLM Answer Quality

Measures whether the generated answer is correct, relevant, and useful.

### RAG Retrieval Quality

Measures whether the system retrieves the correct supporting information before generating an answer.

The overall workflow is:

**Customer Question → Retrieval → Grounded Answer → Evaluation → SQL → Analytics**


# 3. System Architecture

The QualityGuard platform follows this architecture:

```text
Customer Question
       ↓
Question Embedding
       ↓
Semantic Search
       ↓
Relevant Company Policy
       ↓
LLM / Gemini
       ↓
Generated Answer
       ↓
Quality Evaluation
       ↓
PASS / FAIL
       ↓
SQL Database
       ↓
Power BI Dashboard
```

The RAG component helps prevent unsupported answers by providing the LLM with relevant information from an approved knowledge base.

The evaluation layer then measures both retrieval quality and answer quality.


# 4. Technology Stack

| Technology            | Purpose                                   |
| --------------------- | ----------------------------------------- |
| Python                | Core data and AI processing               |
| Pandas                | Dataset manipulation                      |
| Gemini API            | LLM answer generation and evaluation      |
| Sentence Transformers | Text embeddings                           |
| Scikit-learn          | Cosine similarity and evaluation          |
| DuckDB                | SQL analytics and result storage          |
| Pytest                | Automated testing                         |
| Airflow               | Workflow orchestration                    |
| Docker                | Reproducible application environment      |
| Power BI              | Analytics and dashboarding                |
| GitHub                | Version control and project documentation |


In [ ]:
# ============================================
# 5. ENVIRONMENT & API SETUP
# ============================================

import pandas as pd
import numpy as np

from google.colab import userdata
from google import genai

from sklearn.metrics.pairwise import cosine_similarity

import duckdb

print("Libraries imported successfully.")

Libraries imported successfully.


## 5.1 Gemini API Setup

The project uses the Gemini API for two tasks:

1. Generating baseline LLM answers.
2. Evaluating generated answers using an LLM-as-a-judge approach.

The API key is stored securely in Google Colab Secrets rather than being written directly into the notebook.

The secret is named:

`GEMINI_API_KEY`


In [ ]:
# Load Gemini API key securely from Colab Secrets

gemini_api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(
    api_key=gemini_api_key
)

print("Gemini API client configured successfully.")

Gemini API client configured successfully.


# 6. Evaluation Dataset

A reliable LLM evaluation platform needs more than a handful of test questions.

For this project, a synthetic customer-support evaluation dataset is created to simulate a realistic AI evaluation workload.

The dataset is generated from a structured company knowledge base containing multiple customer-support policies.

The evaluation dataset is designed to test:

* Different business categories
* Different question difficulties
* Natural language variations
* Paraphrased questions
* Multi-part questions
* Ambiguous questions
* Unsupported questions
* Retrieval accuracy
* LLM answer quality
* Grounding and hallucination risk

The dataset contains **1,000 evaluation questions** across multiple customer-support categories.

Because this is a synthetic benchmark, the dataset is clearly labelled as simulated data rather than real customer data.


In [ ]:
# ============================================
# 6.1 CREATE KNOWLEDGE BASE
# ============================================

import pandas as pd
import numpy as np

knowledge_base = [
    # -------------------------
    # ORDERS
    # -------------------------
    {
        "policy_id": "ORD001",
        "category": "Orders",
        "title": "Order Cancellation",
        "content": "Orders can be cancelled before they are dispatched."
    },
    {
        "policy_id": "ORD002",
        "category": "Orders",
        "title": "Order Modification",
        "content": "Order details cannot normally be changed after an order has been placed. Customers should contact support as soon as possible if a correction is required."
    },
    {
        "policy_id": "ORD003",
        "category": "Orders",
        "title": "Order Status",
        "content": "Customers can check their order status using the order tracking link provided after dispatch."
    },

    # -------------------------
    # SHIPPING
    # -------------------------
    {
        "policy_id": "SHP001",
        "category": "Shipping",
        "title": "Standard Delivery",
        "content": "Standard delivery usually takes 3 to 5 business days."
    },
    {
        "policy_id": "SHP002",
        "category": "Shipping",
        "title": "Express Delivery",
        "content": "Express delivery usually takes 1 to 2 business days after dispatch."
    },
    {
        "policy_id": "SHP003",
        "category": "Shipping",
        "title": "Order Tracking",
        "content": "Orders can be tracked using the tracking link sent to the customer's email after dispatch."
    },

    # -------------------------
    # RETURNS
    # -------------------------
    {
        "policy_id": "RET001",
        "category": "Returns",
        "title": "Return Window",
        "content": "Items can be returned within 30 days of delivery."
    },
    {
        "policy_id": "RET002",
        "category": "Returns",
        "title": "Original Packaging",
        "content": "Items should normally be returned in their original packaging to be eligible for a return."
    },
    {
        "policy_id": "RET003",
        "category": "Returns",
        "title": "Damaged Items",
        "content": "Customers should contact support promptly if an item arrives damaged."
    },

    # -------------------------
    # REFUNDS
    # -------------------------
    {
        "policy_id": "REF001",
        "category": "Refunds",
        "title": "Refund Processing Time",
        "content": "Refunds are processed within 5 to 7 business days after the return is approved."
    },
    {
        "policy_id": "REF002",
        "category": "Refunds",
        "title": "Refund Payment Method",
        "content": "Refunds are normally returned to the original payment method used for the purchase."
    },

    # -------------------------
    # PAYMENTS
    # -------------------------
    {
        "policy_id": "PAY001",
        "category": "Payments",
        "title": "Accepted Payment Methods",
        "content": "We accept Visa, Mastercard, American Express and PayPal."
    },
    {
        "policy_id": "PAY002",
        "category": "Payments",
        "title": "Payment Failure",
        "content": "If a payment fails, customers should check their payment details and try the transaction again."
    },

    # -------------------------
    # ACCOUNT
    # -------------------------
    {
        "policy_id": "ACC001",
        "category": "Account",
        "title": "Password Reset",
        "content": "Customers can reset their password by selecting 'Forgot password' on the login page and following the instructions sent to their email."
    },
    {
        "policy_id": "ACC002",
        "category": "Account",
        "title": "Email Address Change",
        "content": "Customers can request an email address change through customer support after verifying their account."
    },
    {
        "policy_id": "ACC003",
        "category": "Account",
        "title": "Account Closure",
        "content": "Customers can request account closure through customer support."
    },

    # -------------------------
    # WARRANTY
    # -------------------------
    {
        "policy_id": "WAR001",
        "category": "Warranty",
        "title": "Warranty Period",
        "content": "Products are covered by a 12-month warranty from the date of purchase."
    },
    {
        "policy_id": "WAR002",
        "category": "Warranty",
        "title": "Warranty Claims",
        "content": "Warranty claims can be submitted through customer support with proof of purchase."
    },

    # -------------------------
    # PRODUCTS
    # -------------------------
    {
        "policy_id": "PRO001",
        "category": "Products",
        "title": "Product Availability",
        "content": "Product availability is shown on the product page and may change depending on inventory levels."
    },
    {
        "policy_id": "PRO002",
        "category": "Products",
        "title": "Product Information",
        "content": "Customers can find product specifications and dimensions on the relevant product page."
    },
    {
        "policy_id": "PRO003",
        "category": "Products",
        "title": "Product Compatibility",
        "content": "Customers should check the compatibility information provided on the product page before purchasing."
    },

    # -------------------------
    # DISCOUNTS
    # -------------------------
    {
        "policy_id": "DIS001",
        "category": "Discounts",
        "title": "Promo Codes",
        "content": "Valid promotional codes can be entered during checkout before the order is completed."
    },
    {
        "policy_id": "DIS002",
        "category": "Discounts",
        "title": "Discount Restrictions",
        "content": "Promotional discounts may have specific terms and conditions that should be checked before use."
    },

    # -------------------------
    # SUBSCRIPTIONS
    # -------------------------
    {
        "policy_id": "SUB001",
        "category": "Subscriptions",
        "title": "Subscription Cancellation",
        "content": "Customers can cancel their subscription through their account settings before the next billing date."
    },
    {
        "policy_id": "SUB002",
        "category": "Subscriptions",
        "title": "Subscription Renewal",
        "content": "Subscriptions automatically renew on the scheduled billing date unless cancelled beforehand."
    },

    # -------------------------
    # CUSTOMER SUPPORT
    # -------------------------
    {
        "policy_id": "SUP001",
        "category": "Customer Support",
        "title": "Contact Support",
        "content": "Customers can contact support through the customer support form for assistance with their account or orders."
    },
    {
        "policy_id": "SUP002",
        "category": "Customer Support",
        "title": "Complaints",
        "content": "Customers can submit complaints through the customer support form."
    },

    # -------------------------
    # INVOICES
    # -------------------------
    {
        "policy_id": "INV001",
        "category": "Invoices",
        "title": "Invoice Availability",
        "content": "Invoices are available through the customer's account after an order has been completed."
    },
    {
        "policy_id": "INV002",
        "category": "Invoices",
        "title": "Invoice Requests",
        "content": "Customers can contact support if they require assistance locating an invoice."
    },

    # -------------------------
    # INTERNATIONAL ORDERS
    # -------------------------
    {
        "policy_id": "INT001",
        "category": "International Orders",
        "title": "International Delivery",
        "content": "International delivery times vary depending on the destination and shipping service selected."
    },
    {
        "policy_id": "INT002",
        "category": "International Orders",
        "title": "International Tracking",
        "content": "International orders can be tracked using the tracking information provided after dispatch."
    }
]

knowledge_base_df = pd.DataFrame(knowledge_base)

print("Knowledge base created successfully.")
print(f"Policies: {len(knowledge_base_df)}")
print(f"Categories: {knowledge_base_df['category'].nunique()}")

display(knowledge_base_df)

Knowledge base created successfully.
Policies: 31
Categories: 13


,policy_id,category,title,content
0,ORD001,Orders,Order Cancellation,Orders can be cancelled before they are dispat...
1,ORD002,Orders,Order Modification,Order details cannot normally be changed after...
2,ORD003,Orders,Order Status,Customers can check their order status using t...
3,SHP001,Shipping,Standard Delivery,Standard delivery usually takes 3 to 5 busines...
4,SHP002,Shipping,Express Delivery,Express delivery usually takes 1 to 2 business...
5,SHP003,Shipping,Order Tracking,Orders can be tracked using the tracking link ...
6,RET001,Returns,Return Window,Items can be returned within 30 days of delivery.
7,RET002,Returns,Original Packaging,Items should normally be returned in their ori...
8,RET003,Returns,Damaged Items,Customers should contact support promptly if a...
9,REF001,Refunds,Refund Processing Time,Refunds are processed within 5 to 7 business d...


## 6.2 Validate the Knowledge Base

Before generating evaluation questions, the knowledge base is validated.

Each policy must have:

* A unique policy ID
* A category
* A policy title
* Policy content

This creates a controlled source of truth for the RAG system.


In [ ]:
# ============================================
# KNOWLEDGE BASE VALIDATION
# ============================================

print("Shape:", knowledge_base_df.shape)

print("\nMissing values:")
print(knowledge_base_df.isnull().sum())

print("\nDuplicate policy IDs:")
print(knowledge_base_df["policy_id"].duplicated().sum())

print("\nPolicies by category:")
print(
    knowledge_base_df["category"]
    .value_counts()
)

assert knowledge_base_df["policy_id"].is_unique
assert knowledge_base_df["content"].notna().all()

print("\nKnowledge base validation passed.")

Shape: (31, 4)

Missing values:
policy_id    0
category     0
title        0
content      0
dtype: int64

Duplicate policy IDs:
0

Policies by category:
category
Orders                  3
Shipping                3
Returns                 3
Products                3
Account                 3
Payments                2
Refunds                 2
Warranty                2
Discounts               2
Subscriptions           2
Customer Support        2
Invoices                2
International Orders    2
Name: count, dtype: int64

Knowledge base validation passed.


# ============================================
# 6.3 QUESTION GENERATION TEMPLATES
# ============================================

question_templates = {
    "Refunds": [
        "How long does a refund take?",
        "When should I receive my refund?",
        "How many business days does a refund take?",
        "I returned my item. When will I get my money back?",
        "When will the refund reach me?"
    ],

    "Shipping": [
        "How long does delivery take?",
        "When should my order arrive?",
        "Can I track my order?",
        "Where can I find my tracking information?",
        "How quickly will my order be delivered?"
    ],

    "Returns": [
        "How long do I have to return an item?",
        "Can I return an item?",
        "Does the item need its original packaging?",
        "What should I do if my item arrives damaged?",
        "Can I send this product back?"
    ],

    "Orders": [
        "Can I cancel my order?",
        "Can I change my order?",
        "How can I check my order status?",
        "Can I modify my order after placing it?",
        "Where can I see my order status?"
    ],

    "Payments": [
        "What payment methods do you accept?",
        "Can I pay with Visa?",
        "Can I use PayPal?",
        "What should I do if my payment fails?",
        "Why did my payment fail?"
    ],

    "Account": [
        "How do I reset my password?",
        "I forgot my password. What should I do?",
        "Can I change my email address?",
        "How do I close my account?",
        "Where can I reset my login password?"
    ],

    "Warranty": [
        "How long is the warranty?",
        "How many years is the product covered?",
        "How do I make a warranty claim?",
        "Is my product covered by warranty?",
        "What do I need for a warranty claim?"
    ],

    "Products": [
        "Is this product available?",
        "Where can I find the product specifications?",
        "How can I check product dimensions?",
        "Is this product compatible?",
        "Where can I check compatibility?"
    ],

    "Discounts": [
        "Where do I enter a promo code?",
        "Can I use a discount code?",
        "How do promotional codes work?",
        "Are there restrictions on discounts?",
        "Why isn't my promo code working?"
    ],

    "Subscriptions": [
        "How do I cancel my subscription?",
        "When will my subscription renew?",
        "Can I stop my subscription?",
        "How does subscription renewal work?",
        "How can I cancel before the next payment?"
    ],

    "Customer Support": [
        "How can I contact support?",
        "Where can I submit a complaint?",
        "How do I contact customer service?",
        "Where can I get help with my order?",
        "How do I make a complaint?"
    ],

    "Invoices": [
        "Where can I find my invoice?",
        "How do I get an invoice?",
        "Can I access my invoice online?",
        "Where are my completed order invoices?",
        "I cannot find my invoice. What should I do?"
    ],

    "International Orders": [
        "Do you deliver internationally?",
        "How long does international delivery take?",
        "Can I track an international order?",
        "Where can I find international tracking information?",
        "How are international deliveries handled?"
    ]
}

print("Question templates created.")
print(f"Template categories: {len(question_templates)}")

# ============================================
# 6.4 GENERATE LARGE EVALUATION DATASET
# ============================================

np.random.seed(42)

evaluation_rows = []

# Generate questions based on the knowledge base
for _, policy in knowledge_base_df.iterrows():

    category = policy["category"]

    # Use category-specific templates where available
    templates = question_templates.get(
        category,
        [f"Can you tell me about {policy['title'].lower()}?"]
    )

    for template in templates:

        # Generate multiple controlled variations
        for variation_number in range(1, 9):

            difficulty = np.random.choice(
                ["Easy", "Medium", "Hard"],
                p=[0.35, 0.45, 0.20]
            )

            question_type = np.random.choice(
                [
                    "Direct",
                    "Paraphrased",
                    "Conversational",
                    "Multi-part"
                ],
                p=[0.30, 0.30, 0.25, 0.15]
            )

            evaluation_rows.append({
                "evaluation_id": len(evaluation_rows) + 1,
                "question": template,
                "category": category,
                "policy_id": policy["policy_id"],
                "expected_answer": policy["content"],
                "difficulty": difficulty,
                "question_type": question_type
            })


evaluation_dataset = pd.DataFrame(evaluation_rows)

# Shuffle the dataset
evaluation_dataset = (
    evaluation_dataset
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

# Re-number evaluation IDs
evaluation_dataset["evaluation_id"] = range(
    1,
    len(evaluation_dataset) + 1
)

print("Evaluation dataset created.")
print(f"Rows: {len(evaluation_dataset)}")
print(f"Columns: {len(evaluation_dataset.columns)}")

display(evaluation_dataset.head(20))

This will produce roughly 1,200 rows, because we have 30 policies × 5 templates × 8 variations.

# ============================================
# 6.5 ADD UNSUPPORTED / CHALLENGE QUESTIONS
# ============================================

challenge_questions = [
    {
        "question": "Do you offer student discounts?",
        "category": "Unsupported",
        "policy_id": None,
        "expected_answer": "The available company policies do not contain information about student discounts.",
        "difficulty": "Hard",
        "question_type": "Unsupported"
    },
    {
        "question": "Can I pay using Bitcoin?",
        "category": "Unsupported",
        "policy_id": None,
        "expected_answer": "The available company policies do not contain information about Bitcoin payments.",
        "difficulty": "Hard",
        "question_type": "Unsupported"
    },
    {
        "question": "Do you have physical stores in London?",
        "category": "Unsupported",
        "policy_id": None,
        "expected_answer": "The available company policies do not contain information about physical store locations.",
        "difficulty": "Hard",
        "question_type": "Unsupported"
    },
    {
        "question": "Can I collect my order from your warehouse?",
        "category": "Unsupported",
        "policy_id": None,
        "expected_answer": "The available company policies do not contain information about warehouse collection.",
        "difficulty": "Hard",
        "question_type": "Unsupported"
    },
    {
        "question": "Do you offer a military discount?",
        "category": "Unsupported",
        "policy_id": None,
        "expected_answer": "The available company policies do not contain information about military discounts.",
        "difficulty": "Hard",
        "question_type": "Unsupported"
    }
]

challenge_df = pd.DataFrame(challenge_questions)

challenge_df["evaluation_id"] = range(
    len(evaluation_dataset) + 1,
    len(evaluation_dataset) + len(challenge_df) + 1
)

evaluation_dataset = pd.concat(
    [evaluation_dataset, challenge_df],
    ignore_index=True
)

print("Challenge questions added.")
print(f"Final dataset size: {len(evaluation_dataset)}")

## 6.6 Validate the Evaluation Dataset

The evaluation dataset is now checked before entering the LLM evaluation pipeline.

The validation checks:

* Dataset size
* Duplicate evaluation IDs
* Missing questions
* Category distribution
* Difficulty distribution
* Question-type distribution
* Number of unsupported questions

This ensures that the evaluation dataset is structurally suitable for downstream processing.


# ============================================
# 6.6 DATASET VALIDATION
# ============================================

print("Dataset shape:")
print(evaluation_dataset.shape)

print("\nMissing values:")
print(evaluation_dataset.isnull().sum())

print("\nCategory distribution:")
print(
    evaluation_dataset["category"]
    .value_counts()
)

print("\nDifficulty distribution:")
print(
    evaluation_dataset["difficulty"]
    .value_counts()
)

print("\nQuestion type distribution:")
print(
    evaluation_dataset["question_type"]
    .value_counts()
)

print("\nDuplicate evaluation IDs:")
print(
    evaluation_dataset["evaluation_id"].duplicated().sum()
)

assert evaluation_dataset["evaluation_id"].is_unique
assert evaluation_dataset["question"].notna().all()
assert evaluation_dataset["expected_answer"].notna().all()

print("\nEvaluation dataset validation passed.")

# ============================================
# 6.7 SAVE DATASETS
# ============================================

knowledge_base_df.to_csv(
    "knowledge_base.csv",
    index=False
)

evaluation_dataset.to_csv(
    "evaluation_dataset.csv",
    index=False
)

print("Files saved successfully:")
print("- knowledge_base.csv")
print("- evaluation_dataset.csv")

# ============================================
# 6.8 DATASET SUMMARY
# ============================================

print("=" * 50)
print("QUALITYGUARD DATASET SUMMARY")
print("=" * 50)

print(f"Knowledge-base policies: {len(knowledge_base_df)}")
print(f"Knowledge-base categories: {knowledge_base_df['category'].nunique()}")
print(f"Evaluation questions: {len(evaluation_dataset)}")
print(f"Evaluation categories: {evaluation_dataset['category'].nunique()}")
print(f"Unsupported questions: {(evaluation_dataset['category'] == 'Unsupported').sum()}")

print("\nDifficulty:")
print(
    evaluation_dataset["difficulty"]
    .value_counts()
    .to_dict()
)

print("\nQuestion types:")
print(
    evaluation_dataset["question_type"]
    .value_counts()
    .to_dict()
)

print("\nDataset ready for LLM evaluation.")

In [ ]:
# ============================================
# 6.3–6.8 CORRECTED EVALUATION DATASET
# ============================================

import pandas as pd
import numpy as np

# ------------------------------------------------
# 1. POLICY-SPECIFIC QUESTION GENERATION
# ------------------------------------------------

# Each policy gets questions based specifically on its
# own title/content. This prevents the same question
# being assigned to multiple policies.

question_patterns = [
    "What is the policy for {title}?",
    "Can you explain the {title} policy?",
    "What should I know about {title}?",
    "I need help with {title}. What does the policy say?",
    "Could you tell me about {title}?",
    "What are the rules regarding {title}?",
    "How does the {title} policy work?",
    "What information do you have about {title}?",
    "Can you tell me what the company policy says about {title}?",
    "I have a question about {title}. What is the policy?",
    "What do I need to know regarding {title}?",
    "Please explain the company's {title} policy.",
    "Could you give me more information about {title}?",
    "What are the requirements for {title}?",
    "How does the company handle {title}?"
]


# ------------------------------------------------
# 2. CREATE SUPPORTED QUESTIONS
# ------------------------------------------------

np.random.seed(42)

evaluation_rows = []

for _, policy in knowledge_base_df.iterrows():

    policy_id = policy["policy_id"]
    category = policy["category"]
    title = policy["title"]

    # Create 15 questions for each individual policy
    for question_number, pattern in enumerate(question_patterns, start=1):

        question = pattern.format(
            title=title.lower()
        )

        # Give each question a controlled difficulty
        if question_number <= 5:
            difficulty = "Easy"
        elif question_number <= 11:
            difficulty = "Medium"
        else:
            difficulty = "Hard"

        # Assign question type
        if question_number in [1, 6, 11]:
            question_type = "Direct"
        elif question_number in [2, 7, 12]:
            question_type = "Paraphrased"
        elif question_number in [3, 8, 13]:
            question_type = "Conversational"
        else:
            question_type = "Policy-focused"

        evaluation_rows.append({
            "evaluation_id": len(evaluation_rows) + 1,
            "question": question,
            "category": category,
            "policy_id": policy_id,
            "expected_answer": policy["content"],
            "difficulty": difficulty,
            "question_type": question_type
        })


# ------------------------------------------------
# 3. ADD UNSUPPORTED / OUT-OF-SCOPE QUESTIONS
# ------------------------------------------------

unsupported_questions = [
    "Do you offer student discounts?",
    "Can I pay using Bitcoin?",
    "Do you have physical stores in London?",
    "Can I collect my order directly from your warehouse?",
    "Do you offer a military discount?",
    "Can I pay using Apple Pay?",
    "Do you have a loyalty points programme?",
    "Can I change my delivery address after dispatch?",
    "Do you offer same-day delivery?",
    "Can I exchange an item for a different colour?"
]

for question in unsupported_questions:

    evaluation_rows.append({
        "evaluation_id": len(evaluation_rows) + 1,
        "question": question,
        "category": "Unsupported",
        "policy_id": None,
        "expected_answer": (
            "The available company policy does not contain "
            "enough information to answer this question."
        ),
        "difficulty": "Hard",
        "question_type": "Unsupported"
    })


# ------------------------------------------------
# 4. CREATE DATAFRAME
# ------------------------------------------------

evaluation_dataset = pd.DataFrame(evaluation_rows)


# ------------------------------------------------
# 5. SHUFFLE DATASET
# ------------------------------------------------

evaluation_dataset = (
    evaluation_dataset
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

# Reassign clean sequential IDs after shuffling
evaluation_dataset["evaluation_id"] = range(
    1,
    len(evaluation_dataset) + 1
)


# ------------------------------------------------
# 6. DATA QUALITY CHECKS
# ------------------------------------------------

print("============================================")
print("EVALUATION DATASET VALIDATION")
print("============================================")

print(
    f"Total evaluation questions: "
    f"{len(evaluation_dataset):,}"
)

print(
    f"Supported questions: "
    f"{evaluation_dataset['policy_id'].notna().sum():,}"
)

print(
    f"Unsupported questions: "
    f"{evaluation_dataset['policy_id'].isna().sum():,}"
)

print(
    f"Unique questions: "
    f"{evaluation_dataset['question'].nunique():,}"
)

print(
    f"Unique policy mappings: "
    f"{evaluation_dataset[evaluation_dataset['policy_id'].notna()]['policy_id'].nunique():,}"
)


# ------------------------------------------------
# 7. CHECK FOR DUPLICATE QUESTION → POLICY MAPPINGS
# ------------------------------------------------

duplicate_questions = (
    evaluation_dataset
    .groupby("question")["policy_id"]
    .nunique()
)

ambiguous_questions = duplicate_questions[
    duplicate_questions > 1
]

print(
    f"\nQuestions mapped to multiple policies: "
    f"{len(ambiguous_questions)}"
)


# ------------------------------------------------
# 8. CHECK DUPLICATE QUESTION TEXT
# ------------------------------------------------

duplicate_question_count = (
    evaluation_dataset["question"]
    .duplicated()
    .sum()
)

print(
    f"Duplicate question texts: "
    f"{duplicate_question_count}"
)


# ------------------------------------------------
# 9. VALIDATION ASSERTIONS
# ------------------------------------------------

assert evaluation_dataset["evaluation_id"].is_unique

assert evaluation_dataset["question"].notna().all()

assert evaluation_dataset["expected_answer"].notna().all()

assert len(ambiguous_questions) == 0, (
    "ERROR: Some questions are mapped to multiple policies."
)

assert duplicate_question_count == 0, (
    "ERROR: Duplicate question texts detected."
)


# ------------------------------------------------
# 10. DATASET DISTRIBUTION
# ------------------------------------------------

print("\n============================================")
print("CATEGORY DISTRIBUTION")
print("============================================")

print(
    evaluation_dataset["category"]
    .value_counts()
    .sort_index()
)


print("\n============================================")
print("DIFFICULTY DISTRIBUTION")
print("============================================")

print(
    evaluation_dataset["difficulty"]
    .value_counts()
)


print("\n============================================")
print("QUESTION TYPE DISTRIBUTION")
print("============================================")

print(
    evaluation_dataset["question_type"]
    .value_counts()
)


# ------------------------------------------------
# 11. SAMPLE QUESTIONS
# ------------------------------------------------

print("\n============================================")
print("SAMPLE QUESTIONS")
print("============================================")

display(
    evaluation_dataset[
        [
            "evaluation_id",
            "question",
            "category",
            "policy_id",
            "difficulty",
            "question_type"
        ]
    ].head(15)
)


# ------------------------------------------------
# 12. SAVE DATASETS
# ------------------------------------------------

knowledge_base_df.drop(
    columns=["embedding"],
    errors="ignore"
).to_csv(
    "knowledge_base.csv",
    index=False
)

evaluation_dataset.to_csv(
    "evaluation_dataset.csv",
    index=False
)


# ------------------------------------------------
# 13. FINAL SUMMARY
# ------------------------------------------------

print("\n============================================")
print("DATASET READY")
print("============================================")

print(
    f"Knowledge-base policies: "
    f"{len(knowledge_base_df):,}"
)

print(
    f"Knowledge-base categories: "
    f"{knowledge_base_df['category'].nunique():,}"
)

print(
    f"Evaluation questions: "
    f"{len(evaluation_dataset):,}"
)

print(
    f"Unique questions: "
    f"{evaluation_dataset['question'].nunique():,}"
)

print(
    f"Unsupported questions: "
    f"{(evaluation_dataset['category'] == 'Unsupported').sum():,}"
)

print(
    f"Ambiguous policy mappings: "
    f"{len(ambiguous_questions)}"
)

print(
    f"Duplicate questions: "
    f"{duplicate_question_count}"
)

print("\nDataset successfully created and validated.")

EVALUATION DATASET VALIDATION
Total evaluation questions: 475
Supported questions: 465
Unsupported questions: 10
Unique questions: 475
Unique policy mappings: 31

Questions mapped to multiple policies: 0
Duplicate question texts: 0

CATEGORY DISTRIBUTION
category
Account                 45
Customer Support        30
Discounts               30
International Orders    30
Invoices                30
Orders                  45
Payments                30
Products                45
Refunds                 30
Returns                 45
Shipping                45
Subscriptions           30
Unsupported             10
Warranty                30
Name: count, dtype: int64

DIFFICULTY DISTRIBUTION
difficulty
Medium    186
Easy      155
Hard      134
Name: count, dtype: int64

QUESTION TYPE DISTRIBUTION
question_type
Policy-focused    186
Direct             93
Conversational     93
Paraphrased        93
Unsupported        10
Name: count, dtype: int64

SAMPLE QUESTIONS


,evaluation_id,question,category,policy_id,difficulty,question_type
0,1,What is the policy for contact support?,Customer Support,SUP001,Easy,Direct
1,2,What should I know about invoice requests?,Invoices,INV002,Easy,Conversational
2,3,I have a question about order cancellation. Wh...,Orders,ORD001,Medium,Policy-focused
3,4,What are the requirements for express delivery?,Shipping,SHP002,Hard,Policy-focused
4,5,I have a question about subscription cancellat...,Subscriptions,SUB001,Medium,Policy-focused
5,6,I need help with order status. What does the p...,Orders,ORD003,Easy,Policy-focused
6,7,I need help with order tracking. What does the...,Shipping,SHP003,Easy,Policy-focused
7,8,How does the international delivery policy work?,International Orders,INT001,Medium,Paraphrased
8,9,Could you give me more information about origi...,Returns,RET002,Hard,Conversational
9,10,What is the policy for return window?,Returns,RET001,Easy,Direct



DATASET READY
Knowledge-base policies: 31
Knowledge-base categories: 13
Evaluation questions: 475
Unique questions: 475
Unsupported questions: 10
Ambiguous policy mappings: 0
Duplicate questions: 0

Dataset successfully created and validated.


# 7. Baseline LLM Evaluation

Before introducing Retrieval-Augmented Generation (RAG), the baseline performance of the LLM is measured.

The baseline system sends customer questions directly to the LLM without providing company-specific policy information.

This creates an important comparison:

**Baseline LLM**

Customer Question → LLM → Answer

versus

**RAG System**

Customer Question → Retrieval → Company Policy → LLM → Grounded Answer

The baseline evaluation measures whether an LLM can answer customer-support questions correctly without access to the company's internal knowledge base.

This is important because a general-purpose LLM may produce plausible answers based on general knowledge while still contradicting company-specific policies.

To control API usage, the baseline LLM evaluation uses a representative sample of the full evaluation dataset rather than sending all 1,245 questions to the API.


In [ ]:
# ============================================
# 7.1 CREATE BASELINE EVALUATION SAMPLE
# ============================================

# Number of questions to evaluate with the LLM
BASELINE_SAMPLE_SIZE = 30

# Stratified sampling across category and difficulty
baseline_sample = (
    evaluation_dataset
    .groupby(
        ["category", "difficulty"],
        group_keys=False
    )
    .apply(
        lambda x: x.sample(
            n=min(
                2,
                len(x)
            ),
            random_state=42
        )
    )
    .reset_index(drop=True)
)

# Limit to requested sample size
baseline_sample = (
    baseline_sample
    .sample(
        n=min(BASELINE_SAMPLE_SIZE, len(baseline_sample)),
        random_state=42
    )
    .reset_index(drop=True)
)

print("Baseline evaluation sample created.")
print(f"Sample size: {len(baseline_sample)}")

print("\nCategory distribution:")
print(baseline_sample["category"].value_counts())

print("\nDifficulty distribution:")
print(baseline_sample["difficulty"].value_counts())

display(
    baseline_sample[
        [
            "evaluation_id",
            "question",
            "category",
            "difficulty",
            "question_type"
        ]
    ]
)

Baseline evaluation sample created.
Sample size: 30

Category distribution:
category
Orders                  5
Account                 3
Shipping                3
Products                3
Subscriptions           3
Payments                2
Customer Support        2
International Orders    2
Discounts               2
Returns                 2
Invoices                1
Refunds                 1
Unsupported             1
Name: count, dtype: int64

Difficulty distribution:
difficulty
Medium    12
Easy      11
Hard       7
Name: count, dtype: int64


/tmp/ipykernel_1470/661388205.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,evaluation_id,question,category,difficulty,question_type
0,420,Where can I see my order status?,Orders,Easy,Direct
1,1066,I forgot my password. What should I do?,Account,Easy,Conversational
2,1018,How long does international delivery take?,International Orders,Medium,Paraphrased
3,965,How can I check my order status?,Orders,Easy,Multi-part
4,867,Can I track an international order?,International Orders,Easy,Paraphrased
5,578,Where are my completed order invoices?,Invoices,Medium,Direct
6,1183,How do I contact customer service?,Customer Support,Medium,Direct
7,389,How do I cancel my subscription?,Subscriptions,Medium,Paraphrased
8,429,Where can I reset my login password?,Account,Medium,Direct
9,1025,Why isn't my promo code working?,Discounts,Easy,Multi-part


## 7.2 Generate Baseline LLM Answers

The baseline model receives only the customer's question.

It does **not** receive the company policy.

This allows us to measure how well the LLM performs using its general language knowledge alone.

The function is designed so that API failures do not stop the entire notebook. This is important for production-style workflows because individual API requests can fail due to rate limits, network issues, or temporary service availability.


In [ ]:
# ============================================
# 7.2 BASELINE LLM ANSWER GENERATION
# ============================================

def generate_baseline_answer(question):
    """
    Generate an answer using the LLM without
    providing company-specific policy context.
    """

    try:
        response = client.interactions.create(
            model="gemini-3.6-flash",
            input=question
        )

        return response.output_text.strip()

    except Exception as e:

        print(
            f"API request failed for question: {question}"
        )

        print(
            f"Error: {type(e).__name__}"
        )

        return None

In [ ]:
# ============================================
# 7.3 INITIALISE BASELINE RESULTS
# ============================================

baseline_results = baseline_sample.copy()

baseline_results["actual_answer"] = None
baseline_results["score"] = np.nan
baseline_results["result"] = None
baseline_results["explanation"] = None

print("Baseline evaluation structure created.")

display(baseline_results.head())

Baseline evaluation structure created.


,evaluation_id,question,category,policy_id,expected_answer,difficulty,question_type,actual_answer,score,result,explanation
0,420,Where can I see my order status?,Orders,ORD002,Order details cannot normally be changed after...,Easy,Direct,None,NaN,None,None
1,1066,I forgot my password. What should I do?,Account,ACC003,Customers can request account closure through ...,Easy,Conversational,None,NaN,None,None
2,1018,How long does international delivery take?,International Orders,INT001,International delivery times vary depending on...,Medium,Paraphrased,None,NaN,None,None
3,965,How can I check my order status?,Orders,ORD001,Orders can be cancelled before they are dispat...,Easy,Multi-part,None,NaN,None,None
4,867,Can I track an international order?,International Orders,INT001,International delivery times vary depending on...,Easy,Paraphrased,None,NaN,None,None


In [ ]:
# ============================================
# 7.4 RESUMABLE BASELINE EVALUATION
# ============================================

def run_baseline_generation(results_df):
    """
    Generate baseline answers only for rows
    that do not already contain an answer.
    """

    for i in range(len(results_df)):

        # Skip questions already processed
        if pd.notna(results_df.loc[i, "actual_answer"]):
            continue

        question = results_df.loc[i, "question"]

        answer = generate_baseline_answer(question)

        if answer is None:
            print(
                f"Stopping at row {i + 1} because "
                "the API request failed."
            )
            break

        results_df.loc[i, "actual_answer"] = answer

        print(
            f"Completed {i + 1}/{len(results_df)}"
        )

    return results_df

In [ ]:
# ============================================
# 7.5 SAVE BASELINE PROGRESS
# ============================================

def save_baseline_results(results_df):
    results_df.to_csv(
        "baseline_llm_results.csv",
        index=False
    )

    print(
        "Baseline results saved to "
        "baseline_llm_results.csv"
    )

In [ ]:
# ============================================
# 7.6 RUN BASELINE GENERATION
# ============================================

baseline_results = run_baseline_generation(
    baseline_results
)

save_baseline_results(
    baseline_results
)

Completed 1/30
Completed 2/30
Completed 3/30
Completed 4/30
Completed 5/30
Completed 6/30
Completed 7/30
Completed 8/30
Completed 9/30
Completed 10/30
Completed 11/30
API request failed for question: How can I check my order status?
Error: RateLimitError
Stopping at row 12 because the API request failed.
Baseline results saved to baseline_llm_results.csv


## 7.6 LLM-as-a-Judge

A second LLM evaluation stage is used to assess the quality of the generated answer.

The judge compares the generated answer against the expected company-approved answer.

Scoring:

| Score | Interpretation                    |
| ----- | --------------------------------- |
| 0–3   | Incorrect or seriously misleading |
| 4–6   | Partially correct or incomplete   |
| 7–8   | Mostly correct                    |
| 9–10  | Fully correct and relevant        |

A score of **7 or above** is classified as `PASS`.

A score below 7 is classified as `FAIL`.

The explanation is retained so that evaluation results are interpretable rather than being reduced to a single numerical score.


In [ ]:
# ============================================
# 7.7 LLM-AS-A-JUDGE
# ============================================

def evaluate_answer(
    question,
    expected_answer,
    actual_answer
):
    """
    Evaluate an LLM-generated answer against
    the expected company-approved answer.
    """

    if pd.isna(actual_answer) or actual_answer is None:
        return np.nan, None, None

    judge_prompt = f"""
You are an AI quality evaluator.

Evaluate the AI-generated answer against
the expected company-approved answer.

Question:
{question}

Expected answer:
{expected_answer}

AI-generated answer:
{actual_answer}

Scoring guide:

0-3 = Incorrect or seriously misleading
4-6 = Partially correct or incomplete
7-8 = Mostly correct
9-10 = Fully correct and relevant

Return exactly:

Score: <number>
Explanation: <short explanation>
"""

    try:

        response = client.interactions.create(
            model="gemini-3.6-flash",
            input=judge_prompt
        )

        evaluation_text = response.output_text.strip()

        score_text = (
            evaluation_text
            .split("Score:", 1)[1]
            .split("Explanation:", 1)[0]
            .strip()
        )

        score = int(score_text)

        explanation = (
            evaluation_text
            .split("Explanation:", 1)[1]
            .strip()
        )

        result = (
            "PASS"
            if score >= 7
            else "FAIL"
        )

        return score, result, explanation

    except Exception as e:

        print(
            f"Judge evaluation failed: "
            f"{type(e).__name__}"
        )

        return np.nan, None, None

In [ ]:
# ============================================
# 7.8 RUN LLM-AS-A-JUDGE
# ============================================

for i in range(len(baseline_results)):

    # Skip rows without generated answers
    if pd.isna(
        baseline_results.loc[i, "actual_answer"]
    ):
        continue

    # Skip rows already evaluated
    if pd.notna(
        baseline_results.loc[i, "score"]
    ):
        continue

    question = baseline_results.loc[i, "question"]
    expected = baseline_results.loc[i, "expected_answer"]
    actual = baseline_results.loc[i, "actual_answer"]

    score, result, explanation = evaluate_answer(
        question,
        expected,
        actual
    )

    baseline_results.loc[i, "score"] = score
    baseline_results.loc[i, "result"] = result
    baseline_results.loc[i, "explanation"] = explanation

    print(
        f"Evaluated {i + 1}/{len(baseline_results)}"
    )

save_baseline_results(
    baseline_results
)

print("\nBaseline evaluation completed.")

Judge evaluation failed: RateLimitError
Evaluated 1/30
Judge evaluation failed: RateLimitError
Evaluated 2/30
Judge evaluation failed: RateLimitError
Evaluated 3/30
Judge evaluation failed: RateLimitError
Evaluated 4/30
Judge evaluation failed: RateLimitError
Evaluated 5/30
Judge evaluation failed: RateLimitError
Evaluated 6/30
Judge evaluation failed: RateLimitError
Evaluated 7/30
Judge evaluation failed: RateLimitError
Evaluated 8/30
Judge evaluation failed: RateLimitError
Evaluated 9/30
Judge evaluation failed: RateLimitError
Evaluated 10/30
Judge evaluation failed: RateLimitError
Evaluated 11/30
Baseline results saved to baseline_llm_results.csv

Baseline evaluation completed.


the baseline answers were generated, but the judging stage could not run because Gemini is rate-limited.

In [ ]:
# ============================================
# 7.9 BASELINE PERFORMANCE METRICS
# ============================================

evaluated = baseline_results.dropna(
    subset=["score"]
)

if len(evaluated) > 0:

    average_score = evaluated["score"].mean()

    pass_rate = (
        evaluated["result"]
        .eq("PASS")
        .mean()
        * 100
    )

    fail_rate = (
        evaluated["result"]
        .eq("FAIL")
        .mean()
        * 100
    )

    print("BASELINE LLM PERFORMANCE")
    print("=" * 40)

    print(
        f"Questions evaluated: {len(evaluated)}"
    )

    print(
        f"Average score: {average_score:.2f}/10"
    )

    print(
        f"Pass rate: {pass_rate:.1f}%"
    )

    print(
        f"Fail rate: {fail_rate:.1f}%"
    )

else:

    print(
        "No LLM answers have been evaluated yet."
    )

No LLM answers have been evaluated yet.


In [ ]:
# ============================================
# 7.10 CHECK BASELINE RESULTS
# ============================================

print("Baseline results shape:")
print(baseline_results.shape)

print("\nGenerated answers:")
print(
    baseline_results["actual_answer"]
    .notna()
    .sum()
)

print("\nAnswers missing:")
print(
    baseline_results["actual_answer"]
    .isna()
    .sum()
)

print("\nJudge scores available:")
print(
    baseline_results["score"]
    .notna()
    .sum()
)

display(
    baseline_results[
        [
            "question",
            "category",
            "difficulty",
            "actual_answer",
            "score",
            "result"
        ]
    ].head(15)
)

Baseline results shape:
(30, 11)

Generated answers:
11

Answers missing:
19

Judge scores available:
0


,question,category,difficulty,actual_answer,score,result
0,Where can I see my order status?,Orders,Easy,"To check your order status, it usually depends...",NaN,None
1,I forgot my password. What should I do?,Account,Easy,"To help you get back in, I need a little more ...",NaN,None
2,How long does international delivery take?,International Orders,Medium,International delivery times vary widely depen...,NaN,None
3,How can I check my order status?,Orders,Easy,"To give you the exact steps, **could you tell ...",NaN,None
4,Can I track an international order?,International Orders,Easy,"**Yes, absolutely.** You can track almost all ...",NaN,None
5,Where are my completed order invoices?,Invoices,Medium,"To help you find your invoices, could you let ...",NaN,None
6,How do I contact customer service?,Customer Support,Medium,To help you find the right contact information...,NaN,None
7,How do I cancel my subscription?,Subscriptions,Medium,"To help you cancel your subscription, **I need...",NaN,None
8,Where can I reset my login password?,Account,Medium,"To give you the exact steps, **could you tell ...",NaN,None
9,Why isn't my promo code working?,Discounts,Easy,Because I don't have access to your account or...,NaN,None


We successfully generated a controlled baseline sample, but the free-tier API rate limit prevented automated LLM-as-a-judge scoring. The generated responses were retained for later evaluation.

# 8. SQL Evaluation Database

The evaluation results need to be stored in a structured database so they can be queried, analysed, and connected to reporting tools.

For this project, DuckDB is used as the analytical SQL database.

The database will store information about:

* Customer questions
* Question categories
* Difficulty
* Expected answers
* LLM-generated answers
* Retrieved company policies
* Retrieval similarity
* Retrieval correctness
* LLM evaluation scores
* Hallucination risk
* Faithfulness scores
* Evaluation timestamps

This creates a central source of truth for the QualityGuard platform.

The database can then be queried using SQL and connected to Power BI for reporting.


In [ ]:
# ============================================
# 8.1 CREATE QUALITYGUARD DATABASE
# ============================================

import duckdb

# Create an in-memory DuckDB database
db = duckdb.connect()

print("DuckDB database connected successfully.")

DuckDB database connected successfully.


In [ ]:
# ============================================
# 8.2 PREPARE EVALUATION DATA
# ============================================

sql_dataset = evaluation_dataset.copy()

# Columns that will be populated later
sql_dataset["actual_answer"] = None
sql_dataset["retrieved_policy"] = None
sql_dataset["retrieval_similarity"] = np.nan
sql_dataset["retrieval_correct"] = None
sql_dataset["score"] = np.nan
sql_dataset["result"] = None
sql_dataset["hallucination"] = None
sql_dataset["faithfulness_score"] = np.nan
sql_dataset["explanation"] = None

# Timestamp will be created by DuckDB
print("Evaluation dataset prepared.")
print(f"Rows ready for SQL: {len(sql_dataset)}")

Evaluation dataset prepared.
Rows ready for SQL: 1245


In [ ]:
# ============================================
# 8.3 CREATE SQL TABLE
# ============================================

db.register(
    "project_data",
    sql_dataset
)

db.execute("""
CREATE OR REPLACE TABLE llm_evaluations AS
SELECT
    CAST(evaluation_id AS INTEGER) AS evaluation_id,
    question,
    category,
    difficulty,
    question_type,
    policy_id,
    expected_answer,
    actual_answer,
    retrieved_policy,
    CAST(retrieval_similarity AS DOUBLE)
        AS retrieval_similarity,
    CAST(retrieval_correct AS BOOLEAN)
        AS retrieval_correct,
    CAST(score AS DOUBLE)
        AS score,
    result,
    hallucination,
    CAST(faithfulness_score AS DOUBLE)
        AS faithfulness_score,
    explanation,
    CURRENT_TIMESTAMP AS created_at

FROM project_data
""")

print("Table 'llm_evaluations' created successfully.")

Table 'llm_evaluations' created successfully.


In [ ]:
# ============================================
# 8.4 CHECK SQL TABLE
# ============================================

table_preview = db.execute("""
SELECT *
FROM llm_evaluations
LIMIT 10
""").df()

display(table_preview)

,evaluation_id,question,category,difficulty,question_type,policy_id,expected_answer,actual_answer,retrieved_policy,retrieval_similarity,retrieval_correct,score,result,hallucination,faithfulness_score,explanation,created_at
0,1,When should I receive my refund?,Refunds,Easy,Paraphrased,REF002,Refunds are normally returned to the original ...,<NA>,<NA>,NaN,<NA>,NaN,<NA>,<NA>,NaN,<NA>,2026-09-15 13:20:43.552000+00:00
1,2,Does the item need its original packaging?,Returns,Hard,Direct,RET001,Items can be returned within 30 days of delivery.,<NA>,<NA>,NaN,<NA>,NaN,<NA>,<NA>,NaN,<NA>,2026-09-15 13:20:43.552000+00:00
2,3,Can I return an item?,Returns,Easy,Direct,RET003,Customers should contact support promptly if a...,<NA>,<NA>,NaN,<NA>,NaN,<NA>,<NA>,NaN,<NA>,2026-09-15 13:20:43.552000+00:00
3,4,Can I change my email address?,Account,Medium,Conversational,ACC003,Customers can request account closure through ...,<NA>,<NA>,NaN,<NA>,NaN,<NA>,<NA>,NaN,<NA>,2026-09-15 13:20:43.552000+00:00
4,5,How can I check my order status?,Orders,Easy,Multi-part,ORD003,Customers can check their order status using t...,<NA>,<NA>,NaN,<NA>,NaN,<NA>,<NA>,NaN,<NA>,2026-09-15 13:20:43.552000+00:00
5,6,I returned my item. When will I get my money b...,Refunds,Easy,Direct,REF002,Refunds are normally returned to the original ...,<NA>,<NA>,NaN,<NA>,NaN,<NA>,<NA>,NaN,<NA>,2026-09-15 13:20:43.552000+00:00
6,7,Where can I reset my login password?,Account,Hard,Direct,ACC001,Customers can reset their password by selectin...,<NA>,<NA>,NaN,<NA>,NaN,<NA>,<NA>,NaN,<NA>,2026-09-15 13:20:43.552000+00:00
7,8,How long do I have to return an item?,Returns,Hard,Multi-part,RET001,Items can be returned within 30 days of delivery.,<NA>,<NA>,NaN,<NA>,NaN,<NA>,<NA>,NaN,<NA>,2026-09-15 13:20:43.552000+00:00
8,9,Where can I see my order status?,Orders,Medium,Conversational,ORD002,Order details cannot normally be changed after...,<NA>,<NA>,NaN,<NA>,NaN,<NA>,<NA>,NaN,<NA>,2026-09-15 13:20:43.552000+00:00
9,10,I forgot my password. What should I do?,Account,Hard,Direct,ACC003,Customers can request account closure through ...,<NA>,<NA>,NaN,<NA>,NaN,<NA>,<NA>,NaN,<NA>,2026-09-15 13:20:43.552000+00:00


In [ ]:
# ============================================
# 8.5 TOTAL QUESTIONS
# ============================================

result = db.execute("""
SELECT COUNT(*) AS total_questions
FROM llm_evaluations
""").df()

display(result)

,total_questions
0,1245


In [ ]:
# ============================================
# 8.6 QUESTIONS BY CATEGORY
# ============================================

category_counts = db.execute("""
SELECT
    category,
    COUNT(*) AS question_count
FROM llm_evaluations
GROUP BY category
ORDER BY question_count DESC
""").df()

display(category_counts)

,category,question_count
0,Returns,120
1,Account,120
2,Orders,120
3,Shipping,120
4,Products,120
5,Refunds,80
6,Warranty,80
7,Discounts,80
8,International Orders,80
9,Customer Support,80


In [ ]:
# ============================================
# 8.7 QUESTIONS BY DIFFICULTY
# ============================================

difficulty_counts = db.execute("""
SELECT
    difficulty,
    COUNT(*) AS question_count
FROM llm_evaluations
GROUP BY difficulty
ORDER BY question_count DESC
""").df()

display(difficulty_counts)

,difficulty,question_count
0,Medium,534
1,Easy,464
2,Hard,247


In [ ]:
# ============================================
# 8.8 UNSUPPORTED QUESTIONS
# ============================================

unsupported = db.execute("""
SELECT
    evaluation_id,
    question,
    category,
    difficulty,
    question_type
FROM llm_evaluations
WHERE category = 'Unsupported'
ORDER BY evaluation_id
""").df()

display(unsupported)

,evaluation_id,question,category,difficulty,question_type
0,1241,Do you offer student discounts?,Unsupported,Hard,Unsupported
1,1242,Can I pay using Bitcoin?,Unsupported,Hard,Unsupported
2,1243,Do you have physical stores in London?,Unsupported,Hard,Unsupported
3,1244,Can I collect my order from your warehouse?,Unsupported,Hard,Unsupported
4,1245,Do you offer a military discount?,Unsupported,Hard,Unsupported


## 8.9 QualityGuard Reporting View

A SQL view provides a simplified reporting layer for downstream analytics.

Power BI will eventually connect to this view rather than needing to understand the entire underlying database structure.

This creates a separation between:

**Raw evaluation data → SQL transformation → Reporting layer → Power BI**


In [ ]:
# ============================================
# 8.9 CREATE POWER BI REPORTING VIEW
# ============================================

db.execute("""
CREATE OR REPLACE VIEW qualityguard_dashboard AS

SELECT
    evaluation_id,
    category,
    difficulty,
    question_type,
    question,
    retrieval_similarity,
    retrieval_correct,
    score,
    result,
    hallucination,
    faithfulness_score,
    created_at

FROM llm_evaluations
""")

print("Reporting view created successfully.")

Reporting view created successfully.


In [ ]:
# ============================================
# 8.10 TEST REPORTING VIEW
# ============================================

dashboard_data = db.execute("""
SELECT *
FROM qualityguard_dashboard
LIMIT 10
""").df()

display(dashboard_data)

,evaluation_id,category,difficulty,question_type,question,retrieval_similarity,retrieval_correct,score,result,hallucination,faithfulness_score,created_at
0,1,Refunds,Easy,Paraphrased,When should I receive my refund?,NaN,<NA>,NaN,<NA>,<NA>,NaN,2026-09-15 13:20:43.552000+00:00
1,2,Returns,Hard,Direct,Does the item need its original packaging?,NaN,<NA>,NaN,<NA>,<NA>,NaN,2026-09-15 13:20:43.552000+00:00
2,3,Returns,Easy,Direct,Can I return an item?,NaN,<NA>,NaN,<NA>,<NA>,NaN,2026-09-15 13:20:43.552000+00:00
3,4,Account,Medium,Conversational,Can I change my email address?,NaN,<NA>,NaN,<NA>,<NA>,NaN,2026-09-15 13:20:43.552000+00:00
4,5,Orders,Easy,Multi-part,How can I check my order status?,NaN,<NA>,NaN,<NA>,<NA>,NaN,2026-09-15 13:20:43.552000+00:00
5,6,Refunds,Easy,Direct,I returned my item. When will I get my money b...,NaN,<NA>,NaN,<NA>,<NA>,NaN,2026-09-15 13:20:43.552000+00:00
6,7,Account,Hard,Direct,Where can I reset my login password?,NaN,<NA>,NaN,<NA>,<NA>,NaN,2026-09-15 13:20:43.552000+00:00
7,8,Returns,Hard,Multi-part,How long do I have to return an item?,NaN,<NA>,NaN,<NA>,<NA>,NaN,2026-09-15 13:20:43.552000+00:00
8,9,Orders,Medium,Conversational,Where can I see my order status?,NaN,<NA>,NaN,<NA>,<NA>,NaN,2026-09-15 13:20:43.552000+00:00
9,10,Account,Hard,Direct,I forgot my password. What should I do?,NaN,<NA>,NaN,<NA>,<NA>,NaN,2026-09-15 13:20:43.552000+00:00


In [ ]:
# ============================================
# 8.11 DATABASE SUMMARY
# ============================================

summary = db.execute("""
SELECT
    COUNT(*) AS total_questions,

    COUNT(DISTINCT category)
        AS total_categories,

    COUNT(*) FILTER (
        WHERE category = 'Unsupported'
    ) AS unsupported_questions,

    COUNT(*) FILTER (
        WHERE score IS NOT NULL
    ) AS llm_evaluated_questions,

    COUNT(*) FILTER (
        WHERE retrieval_similarity IS NOT NULL
    ) AS retrieval_evaluated_questions

FROM llm_evaluations
""").df()

display(summary)

,total_questions,total_categories,unsupported_questions,llm_evaluated_questions,retrieval_evaluated_questions
0,1245,14,5,0,0


# 9. Semantic Retrieval Evaluation

The retrieval system is responsible for finding the company policy that is most relevant to a customer's question.

For example:

**Customer question:**

> "I sent my item back. When will I get my money?"

The system should recognise that this question is related to the **Refunds** policy, even though the wording does not exactly match the policy.

To achieve this, the system converts both questions and policies into numerical representations called **embeddings**.

The system then calculates the similarity between the question embedding and each policy embedding.

The policy with the highest similarity becomes the top retrieval result.

This allows the system to perform semantic search rather than relying only on exact keyword matching.

The retrieval evaluation will measure:

* Top-1 retrieval accuracy
* Retrieval similarity
* Correct and incorrect retrievals
* Retrieval performance by category
* Retrieval performance by difficulty
* Unsupported-question behaviour

The full evaluation dataset will be used, rather than only the small LLM sample.


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [ ]:
# ============================================
# 9.1 CHECK EMBEDDING MODEL
# ============================================

print(
    "Embedding model:",
    embedding_model
)

Embedding model: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


In [ ]:
# ============================================
# CREATE POLICY EMBEDDINGS
# ============================================

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

knowledge_base_df["embedding"] = (
    knowledge_base_df["content"]
    .apply(
        lambda text: embedding_model.encode(text)
    )
)

print("Policy embeddings created successfully.")
print(
    f"Policies embedded: {len(knowledge_base_df)}"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Policy embeddings created successfully.
Policies embedded: 31


In [ ]:
# ============================================
# 9.3 SEMANTIC RETRIEVAL FUNCTION
# ============================================

def retrieve_policy(
    question,
    knowledge_base_df,
    embedding_model,
    threshold=0.30
):
    """
    Retrieve the most semantically similar
    company policy for a customer question.

    Returns:
        The best matching policy if its similarity
        is above the threshold.
        Otherwise returns None.
    """

    # Convert the customer question into an embedding
    question_embedding = embedding_model.encode(
        question
    )

    # Calculate similarity between the question
    # and every policy in the knowledge base
    similarities = []

    for embedding in knowledge_base_df["embedding"]:

        similarity = cosine_similarity(
            [question_embedding],
            [embedding]
        )[0][0]

        similarities.append(similarity)

    # Create a copy so the original knowledge base
    # is not modified
    results = knowledge_base_df.copy()

    results["similarity"] = similarities

    # Highest similarity = best semantic match
    results = results.sort_values(
        "similarity",
        ascending=False
    )

    # Select the best matching policy
    best_match = results.iloc[0]

    # Reject the result if similarity is too low
    if best_match["similarity"] < threshold:
        return None

    return best_match

In [ ]:
# ============================================
# 9.4 TEST RETRIEVAL
# ============================================

test_question = (
    "I returned my item. "
    "When will I receive my money?"
)

retrieved_policy = retrieve_policy(
    test_question,
    knowledge_base_df,
    embedding_model
)

if retrieved_policy is not None:

    print("Question:")
    print(test_question)

    print("\nRetrieved policy:")
    print(retrieved_policy["title"])

    print("\nPolicy ID:")
    print(retrieved_policy["policy_id"])

    print("\nSimilarity:")
    print(
        round(
            retrieved_policy["similarity"],
            4
        )
    )

    print("\nPolicy content:")
    print(retrieved_policy["content"])

else:
    print("No sufficiently similar policy found.")

Question:
I returned my item. When will I receive my money?

Retrieved policy:
Return Window

Policy ID:
RET001

Similarity:
0.6676

Policy content:
Items can be returned within 30 days of delivery.


In [ ]:
# ============================================
# 9.5 INSPECT TOP RETRIEVAL MATCHES
# ============================================

question_embedding = embedding_model.encode(
    test_question
)

retrieval_debug = knowledge_base_df.copy()

retrieval_debug["similarity"] = (
    retrieval_debug["embedding"].apply(
        lambda embedding: cosine_similarity(
            [question_embedding],
            [embedding]
        )[0][0]
    )
)

top_matches = (
    retrieval_debug[
        [
            "policy_id",
            "title",
            "content",
            "similarity"
        ]
    ]
    .sort_values(
        "similarity",
        ascending=False
    )
    .head(5)
    .reset_index(drop=True)
)

top_matches["similarity"] = (
    top_matches["similarity"].round(4)
)

display(top_matches)

,policy_id,title,content,similarity
0,RET001,Return Window,Items can be returned within 30 days of delivery.,0.6676
1,REF001,Refund Processing Time,Refunds are processed within 5 to 7 business d...,0.5450
2,RET002,Original Packaging,Items should normally be returned in their ori...,0.5138
3,REF002,Refund Payment Method,Refunds are normally returned to the original ...,0.5137
4,RET003,Damaged Items,Customers should contact support promptly if a...,0.3946


The question contains strong language about returning an item, so the embedding model considers the Return Window policy semantically close, even though the user's actual intent is about the refund timeline.

Baseline semantic retrieval can confuse closely related business intents, motivating retrieval evaluation and further optimisation.

In [ ]:
# ============================================
# 9.6 FULL RETRIEVAL EVALUATION
# ============================================

retrieval_results = evaluation_dataset.copy()

retrieval_results["retrieved_policy_id"] = None
retrieval_results["retrieved_policy"] = None
retrieval_results["retrieval_similarity"] = np.nan
retrieval_results["retrieval_correct"] = False

for i in range(len(retrieval_results)):

    question = retrieval_results.loc[i, "question"]

    retrieved = retrieve_policy(
        question,
        knowledge_base_df,
        embedding_model
    )

    if retrieved is None:
        continue

    retrieval_results.loc[i, "retrieved_policy_id"] = (
        retrieved["policy_id"]
    )

    retrieval_results.loc[i, "retrieved_policy"] = (
        retrieved["content"]
    )

    retrieval_results.loc[i, "retrieval_similarity"] = (
        retrieved["similarity"]
    )

    expected_policy_id = retrieval_results.loc[
        i,
        "policy_id"
    ]

    # Only supported questions have a known
    # correct policy ID
    if pd.notna(expected_policy_id):

        retrieval_results.loc[
            i,
            "retrieval_correct"
        ] = (
            retrieved["policy_id"]
            == expected_policy_id
        )


# --------------------------------------------
# EVALUATE SUPPORTED QUESTIONS
# --------------------------------------------

supported_questions = retrieval_results[
    retrieval_results["policy_id"].notna()
]

retrieval_accuracy = (
    supported_questions["retrieval_correct"].mean()
    * 100
)

average_similarity = (
    supported_questions["retrieval_similarity"].mean()
)


print("============================================")
print("RETRIEVAL EVALUATION RESULTS")
print("============================================")

print(
    f"Supported questions: "
    f"{len(supported_questions):,}"
)

print(
    f"Retrieval accuracy: "
    f"{retrieval_accuracy:.2f}%"
)

print(
    f"Average similarity: "
    f"{average_similarity:.4f}"
)

RETRIEVAL EVALUATION RESULTS
Supported questions: 465
Retrieval accuracy: 84.30%
Average similarity: 0.6232


In [ ]:
# ============================================
# 9.7 LOWEST-CONFIDENCE RETRIEVALS
# ============================================

lowest_confidence = (
    supported_questions[
        [
            "question",
            "category",
            "policy_id",
            "retrieved_policy_id",
            "retrieval_similarity",
            "retrieval_correct"
        ]
    ]
    .sort_values(
        "retrieval_similarity"
    )
    .head(20)
)

display(lowest_confidence)

,question,category,policy_id,retrieved_policy_id,retrieval_similarity,retrieval_correct
15,Can you explain the product information policy?,Products,PRO002,DIS002,0.358909,False
247,How does the product information policy work?,Products,PRO002,WAR001,0.376929,False
218,What are the requirements for return window?,Returns,RET001,RET002,0.381090,False
236,What should I know about return window?,Returns,RET001,RET002,0.381986,False
123,Could you tell me about return window?,Returns,RET001,REF002,0.396076,False
287,Please explain the company's product informati...,Products,PRO002,WAR001,0.396260,False
338,What do I need to know regarding return window?,Returns,RET001,RET002,0.398822,False
474,Could you give me more information about retur...,Returns,RET001,REF002,0.403759,False
450,Can you tell me what the company policy says a...,Products,PRO002,WAR001,0.415332,False
101,Can you explain the standard delivery policy?,Shipping,SHP001,SHP001,0.418323,True


In [ ]:
# ============================================
# 9.8 RETRIEVAL ERROR ANALYSIS
# ============================================

# ------------------------------------------------
# 1. ALL INCORRECT RETRIEVALS
# ------------------------------------------------

retrieval_errors = (
    supported_questions[
        supported_questions["retrieval_correct"] == False
    ]
    [
        [
            "question",
            "category",
            "policy_id",
            "retrieved_policy_id",
            "retrieval_similarity"
        ]
    ]
    .sort_values("retrieval_similarity")
)

print("============================================")
print("RETRIEVAL ERROR SUMMARY")
print("============================================")

print(
    f"Total supported questions: "
    f"{len(supported_questions):,}"
)

print(
    f"Incorrect retrievals: "
    f"{len(retrieval_errors):,}"
)

print(
    f"Retrieval error rate: "
    f"{(len(retrieval_errors) / len(supported_questions)) * 100:.2f}%"
)

print("\n============================================")
print("MOST COMMON RETRIEVAL ERRORS")
print("============================================")

error_pairs = (
    retrieval_errors
    .groupby(
        [
            "policy_id",
            "retrieved_policy_id"
        ]
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False
    )
)

display(error_pairs.head(15))

print("\n============================================")
print("ERRORS BY CATEGORY")
print("============================================")

category_errors = (
    retrieval_errors
    .groupby("category")
    .size()
    .reset_index(name="incorrect_retrievals")
    .sort_values(
        "incorrect_retrievals",
        ascending=False
    )
)

display(category_errors)

print("\n============================================")
print("LOWEST-CONFIDENCE ERRORS")
print("============================================")

display(
    retrieval_errors.head(20)
)

RETRIEVAL ERROR SUMMARY
Total supported questions: 465
Incorrect retrievals: 73
Retrieval error rate: 15.70%

MOST COMMON RETRIEVAL ERRORS


,policy_id,retrieved_policy_id,count
0,INV001,INV002,15
8,RET001,RET002,10
9,SHP003,INT002,8
11,WAR001,WAR002,8
3,PAY001,REF002,6
5,PRO002,WAR001,6
7,RET001,REF002,5
1,ORD002,ORD001,4
2,ORD003,ORD001,4
10,SUP001,RET003,4



ERRORS BY CATEGORY


,category,incorrect_retrievals
1,Invoices,15
6,Returns,15
8,Warranty,8
7,Shipping,8
2,Orders,8
4,Products,7
3,Payments,6
0,Customer Support,4
5,Refunds,2



LOWEST-CONFIDENCE ERRORS


,question,category,policy_id,retrieved_policy_id,retrieval_similarity
15,Can you explain the product information policy?,Products,PRO002,DIS002,0.358909
247,How does the product information policy work?,Products,PRO002,WAR001,0.376929
218,What are the requirements for return window?,Returns,RET001,RET002,0.381090
236,What should I know about return window?,Returns,RET001,RET002,0.381986
123,Could you tell me about return window?,Returns,RET001,REF002,0.396076
287,Please explain the company's product informati...,Products,PRO002,WAR001,0.396260
338,What do I need to know regarding return window?,Returns,RET001,RET002,0.398822
474,Could you give me more information about retur...,Returns,RET001,REF002,0.403759
450,Can you tell me what the company policy says a...,Products,PRO002,WAR001,0.415332
248,What information do you have about return window?,Returns,RET001,RET002,0.421027


The biggest finding

INV001 → INV002 accounts for 15 of the 73 errors, or about 20.5% of all retrieval failures.

And RET001 → RET002 accounts for another 10 errors.

So two policy pairs alone explain a substantial portion of the retrieval weakness.

# 10. RAG: Grounded Answer Generation

Retrieval-Augmented Generation (RAG) combines two stages:

1. **Retrieval** — find the most relevant company policy.
2. **Generation** — ask the LLM to answer using the retrieved policy.

The key difference between the baseline LLM and the RAG system is that the RAG system is instructed to use trusted company information rather than relying only on its general knowledge.

The RAG prompt also instructs the model not to invent information.

If the retrieved policy does not contain enough information, the model should explicitly state that it does not have enough information.

In [ ]:
# ============================================
# 10.1 BUILD RAG PROMPT
# ============================================

def build_rag_prompt(question, retrieved_policy):
    """
    Build a grounded prompt for the RAG system.

    The LLM is instructed to answer using only
    the retrieved company policy.
    """

    prompt = f"""
You are a customer support assistant.

Answer the customer's question using ONLY the company
policy provided below.

COMPANY POLICY:
{retrieved_policy}

CUSTOMER QUESTION:
{question}

RULES:
1. Use only information contained in the company policy.
2. Do not invent or assume additional information.
3. Do not use general knowledge to fill missing details.
4. If the policy does not contain enough information to answer
   the question, say that you do not have enough information.
5. Keep the answer clear and concise.

Provide the final answer to the customer.
"""

    return prompt.strip()


print("RAG prompt builder created successfully.")

RAG prompt builder created successfully.


In [ ]:
# ============================================
# 10.2 TEST RAG PROMPT
# ============================================

test_question = (
    "What happens if I want to return an item?"
)

retrieved = retrieve_policy(
    test_question,
    knowledge_base_df,
    embedding_model
)

print("============================================")
print("RAG TEST")
print("============================================")

print(f"Question: {test_question}")
print(f"Retrieved policy: {retrieved['title']}")
print(f"Policy ID: {retrieved['policy_id']}")
print(f"Similarity: {retrieved['similarity']:.4f}")

rag_prompt = build_rag_prompt(
    test_question,
    retrieved["content"]
)

print("\n============================================")
print("GENERATED RAG PROMPT")
print("============================================")

print(rag_prompt)

RAG TEST
Question: What happens if I want to return an item?
Retrieved policy: Original Packaging
Policy ID: RET002
Similarity: 0.8022

GENERATED RAG PROMPT
You are a customer support assistant.

Answer the customer's question using ONLY the company
policy provided below.

COMPANY POLICY:
Items should normally be returned in their original packaging to be eligible for a return.

CUSTOMER QUESTION:
What happens if I want to return an item?

RULES:
1. Use only information contained in the company policy.
2. Do not invent or assume additional information.
3. Do not use general knowledge to fill missing details.
4. If the policy does not contain enough information to answer
   the question, say that you do not have enough information.
5. Keep the answer clear and concise.

Provide the final answer to the customer.


## 10.3 Grounded Answer Generation

The retrieved policy is passed to the Gemini model as context.

The model is instructed to:

- use only the retrieved policy
- avoid unsupported claims
- avoid filling gaps with general knowledge
- explicitly state when the policy is insufficient

Because the Gemini API has a limited development quota, the generation stage is executed only on a controlled sample rather than the full evaluation dataset.

In [ ]:
# ============================================
# 10.3 GROUNDED RAG ANSWER GENERATION
# ============================================

def generate_rag_answer(question, retrieved_policy):
    """
    Generate an answer using only the retrieved
    company policy.

    Returns:
        Generated answer if successful.
        None if the API quota is unavailable.
    """

    rag_prompt = build_rag_prompt(
        question,
        retrieved_policy
    )

    try:

        response = client.interactions.create(
            model="gemini-3.6-flash",
            input=rag_prompt
        )

        return response.output_text.strip()

    except Exception as e:

        print("Gemini API request could not be completed.")
        print(f"Reason: {type(e).__name__}")

        return None


# ------------------------------------------------
# CONTROLLED TEST
# ------------------------------------------------

test_question = (
    "What happens if I want to return an item?"
)

retrieved = retrieve_policy(
    test_question,
    knowledge_base_df,
    embedding_model
)

print("============================================")
print("RAG GENERATION TEST")
print("============================================")

print(f"Question: {test_question}")
print(f"Retrieved policy: {retrieved['title']}")
print(f"Policy ID: {retrieved['policy_id']}")
print(f"Similarity: {retrieved['similarity']:.4f}")

generated_answer = generate_rag_answer(
    test_question,
    retrieved["content"]
)

print("\n============================================")
print("RAG ANSWER")
print("============================================")

if generated_answer is not None:
    print(generated_answer)
else:
    print("No answer generated because the Gemini API was unavailable.")

RAG GENERATION TEST
Question: What happens if I want to return an item?
Retrieved policy: Original Packaging
Policy ID: RET002
Similarity: 0.8022

RAG ANSWER
To be eligible for a return, your item should normally be returned in its original packaging.


In [ ]:
# ============================================
# 10.4 CONTROLLED RAG EVALUATION
# ============================================

# Select a small controlled sample.
# We deliberately avoid running this across all 475 questions
# because Gemini API usage is limited.

sample_size = 5

rag_sample = (
    evaluation_dataset[
        evaluation_dataset["policy_id"].notna()
    ]
    .sample(
        n=sample_size,
        random_state=42
    )
    .reset_index(drop=True)
)

rag_results = []

print("============================================")
print("CONTROLLED RAG EVALUATION")
print("============================================")

for i, row in rag_sample.iterrows():

    question = row["question"]
    expected_policy_id = row["policy_id"]

    retrieved = retrieve_policy(
        question,
        knowledge_base_df,
        embedding_model
    )

    if retrieved is None:

        rag_results.append({
            "evaluation_id": row["evaluation_id"],
            "question": question,
            "expected_policy_id": expected_policy_id,
            "retrieved_policy_id": None,
            "retrieval_similarity": np.nan,
            "retrieval_correct": False,
            "generated_answer": None
        })

        continue

    retrieved_policy_id = retrieved["policy_id"]

    retrieval_correct = (
        retrieved_policy_id == expected_policy_id
    )

    generated_answer = generate_rag_answer(
        question,
        retrieved["content"]
    )

    rag_results.append({
        "evaluation_id": row["evaluation_id"],
        "question": question,
        "expected_policy_id": expected_policy_id,
        "retrieved_policy_id": retrieved_policy_id,
        "retrieval_similarity": retrieved["similarity"],
        "retrieval_correct": retrieval_correct,
        "generated_answer": generated_answer
    })

    print(f"\nQuestion {i + 1}/{sample_size}")
    print(f"Question: {question}")
    print(f"Expected policy: {expected_policy_id}")
    print(f"Retrieved policy: {retrieved_policy_id}")
    print(f"Similarity: {retrieved['similarity']:.4f}")
    print(f"Retrieval correct: {retrieval_correct}")

    if generated_answer:
        print(f"RAG answer: {generated_answer}")
    else:
        print("RAG answer: Not generated")


rag_results_df = pd.DataFrame(rag_results)

print("\n============================================")
print("RAG SAMPLE COMPLETE")
print("============================================")

print(
    f"Questions evaluated: "
    f"{len(rag_results_df)}"
)

print(
    f"Retrieval accuracy: "
    f"{rag_results_df['retrieval_correct'].mean() * 100:.2f}%"
)

print(
    f"Average retrieval similarity: "
    f"{rag_results_df['retrieval_similarity'].mean():.4f}"
)

display(rag_results_df)

CONTROLLED RAG EVALUATION
Gemini API request could not be completed.
Reason: RateLimitError

Question 1/5
Question: How does the company handle invoice availability?
Expected policy: INV001
Retrieved policy: INV002
Similarity: 0.7108
Retrieval correct: False
RAG answer: Not generated
Gemini API request could not be completed.
Reason: RateLimitError

Question 2/5
Question: Can you explain the promo codes policy?
Expected policy: DIS001
Retrieved policy: DIS001
Similarity: 0.6855
Retrieval correct: True
RAG answer: Not generated
Gemini API request could not be completed.
Reason: RateLimitError

Question 3/5
Question: What information do you have about accepted payment methods?
Expected policy: PAY001
Retrieved policy: PAY001
Similarity: 0.5221
Retrieval correct: True
RAG answer: Not generated
Gemini API request could not be completed.
Reason: RateLimitError

Question 4/5
Question: Can you explain the damaged items policy?
Expected policy: RET003
Retrieved policy: RET003
Similarity: 0.522

,evaluation_id,question,expected_policy_id,retrieved_policy_id,retrieval_similarity,retrieval_correct,generated_answer
0,57,How does the company handle invoice availability?,INV001,INV002,0.710793,False,None
1,80,Can you explain the promo codes policy?,DIS001,DIS001,0.685473,True,None
2,35,What information do you have about accepted pa...,PAY001,PAY001,0.522135,True,None
3,466,Can you explain the damaged items policy?,RET003,RET003,0.522852,True,None
4,304,Can you tell me what the company policy says a...,SUB001,SUB001,0.729983,True,None


RAG prompt construction works.

One live RAG generation was successfully demonstrated.

The Gemini quota prevented further generation.

Retrieval can still be evaluated independently.

The controlled retrieval sample achieved 80%.

Full-scale generation/faithfulness evaluation is deferred until API capacity is available.



# 11. Faithfulness & Hallucination Detection

A good RAG system should not only retrieve relevant information.

The generated answer should also remain supported by the retrieved policy.

This section evaluates whether an answer contains information that appears to be supported by its retrieved evidence.

Because the Gemini API development quota has been reached, the project uses a lightweight rule-based hallucination detector as a baseline.

This is not a replacement for LLM-as-a-judge evaluation.

Instead, it provides:

- a reproducible baseline
- a zero-API-cost evaluation method
- a simple risk signal
- a foundation for future improvements

The final system can later compare this baseline against an LLM-based faithfulness evaluator.

In [ ]:
# ============================================
# 11.1 RULE-BASED HALLUCINATION BASELINE
# ============================================

import re


def check_hallucination(answer, policy):
    """
    Estimate hallucination risk using lexical overlap.

    LOW  = substantial overlap between answer and policy
    HIGH = limited overlap between answer and policy

    This is a simple baseline and should not be treated
    as a definitive hallucination detector.
    """

    if answer is None or pd.isna(answer):
        return "UNKNOWN"

    answer_words = set(
        re.findall(
            r"\b[a-zA-Z]+\b",
            answer.lower()
        )
    )

    policy_words = set(
        re.findall(
            r"\b[a-zA-Z]+\b",
            policy.lower()
        )
    )

    if len(answer_words) == 0:
        return "UNKNOWN"

    common_words = answer_words.intersection(
        policy_words
    )

    overlap = (
        len(common_words)
        / len(answer_words)
    )

    if overlap >= 0.50:
        return "LOW"

    return "HIGH"


print("Hallucination baseline created successfully.")

Hallucination baseline created successfully.


In [ ]:
# ============================================
# 11.2 TEST HALLUCINATION DETECTOR
# ============================================

test_answer = (
    "To be eligible for a return, your item should "
    "normally be returned in its original packaging."
)

test_policy = (
    "Items should normally be returned in their "
    "original packaging to be eligible for a return."
)

hallucination_result = check_hallucination(
    test_answer,
    test_policy
)

print("============================================")
print("HALLUCINATION DETECTION TEST")
print("============================================")

print(f"Answer: {test_answer}")
print(f"Policy: {test_policy}")
print(f"Hallucination risk: {hallucination_result}")

HALLUCINATION DETECTION TEST
Answer: To be eligible for a return, your item should normally be returned in its original packaging.
Policy: Items should normally be returned in their original packaging to be eligible for a return.
Hallucination risk: LOW


In [ ]:
# ============================================
# 11.3 HALLUCINATION BASELINE ON AVAILABLE ANSWERS
# ============================================

# Start with the successful RAG example
available_answers = pd.DataFrame([
    {
        "question": "What happens if I want to return an item?",
        "retrieved_policy": (
            "Items should normally be returned in their "
            "original packaging to be eligible for a return."
        ),
        "generated_answer": (
            "To be eligible for a return, your item should "
            "normally be returned in its original packaging."
        )
    }
])

# Apply the hallucination detector
available_answers["hallucination"] = (
    available_answers.apply(
        lambda row: check_hallucination(
            row["generated_answer"],
            row["retrieved_policy"]
        ),
        axis=1
    )
)

print("============================================")
print("HALLUCINATION BASELINE RESULTS")
print("============================================")

display(available_answers)

print("\n============================================")
print("RISK SUMMARY")
print("============================================")

print(
    available_answers["hallucination"]
    .value_counts()
)

HALLUCINATION BASELINE RESULTS


,question,retrieved_policy,generated_answer,hallucination
0,What happens if I want to return an item?,Items should normally be returned in their ori...,"To be eligible for a return, your item should ...",LOW



RISK SUMMARY
hallucination
LOW    1
Name: count, dtype: int64


# 12. End-to-End QualityGuard Pipeline

The QualityGuard pipeline combines the main evaluation components developed throughout the project.

For each evaluation question, the pipeline:

1. Retrieves the most relevant company policy.
2. Calculates semantic similarity.
3. Checks whether the correct policy was retrieved.
4. Evaluates hallucination risk when an answer is available.
5. Assigns an overall evaluation status.

The pipeline separates retrieval evaluation from answer evaluation.

This is important because a system can retrieve the wrong policy even when the LLM generates a well-written answer.

Gemini-generated answers are only evaluated when they are available. Missing answers are recorded as `UNKNOWN` rather than being treated as failures.

In [ ]:
# ============================================
# 12.1 END-TO-END QUALITYGUARD PIPELINE
# ============================================

def run_qualityguard_pipeline(
    evaluation_dataset,
    knowledge_base_df,
    embedding_model
):
    """
    Run the non-API portion of the QualityGuard pipeline.

    Steps:
        1. Retrieve relevant policy
        2. Calculate similarity
        3. Check retrieval correctness
        4. Check hallucination risk when an answer exists
        5. Assign evaluation status
    """

    results = []

    for _, row in evaluation_dataset.iterrows():

        question = row["question"]
        expected_policy_id = row["policy_id"]

        # ----------------------------------------
        # STEP 1: RETRIEVAL
        # ----------------------------------------

        retrieved = retrieve_policy(
            question,
            knowledge_base_df,
            embedding_model
        )

        if retrieved is None:

            results.append({
                "evaluation_id": row["evaluation_id"],
                "question": question,
                "category": row["category"],
                "difficulty": row["difficulty"],
                "expected_policy_id": expected_policy_id,
                "retrieved_policy_id": None,
                "retrieved_policy": None,
                "retrieval_similarity": np.nan,
                "retrieval_correct": False,
                "generated_answer": None,
                "hallucination": "UNKNOWN",
                "evaluation_status": "NO_RETRIEVAL"
            })

            continue

        # ----------------------------------------
        # STEP 2: RETRIEVAL EVALUATION
        # ----------------------------------------

        retrieved_policy_id = retrieved["policy_id"]

        if pd.notna(expected_policy_id):

            retrieval_correct = (
                retrieved_policy_id
                == expected_policy_id
            )

        else:

            # Unsupported questions should ideally
            # not retrieve a policy.
            retrieval_correct = False

        # ----------------------------------------
        # STEP 3: ANSWER AVAILABILITY
        # ----------------------------------------

        generated_answer = None

        hallucination = "UNKNOWN"

        # We do not call Gemini here.
        # Answers can be added later when API capacity
        # becomes available.

        # ----------------------------------------
        # STEP 4: OVERALL STATUS
        # ----------------------------------------

        if pd.isna(expected_policy_id):

            if retrieved is None:
                evaluation_status = "SUPPORTED_BY_NONE"
            else:
                evaluation_status = "UNSUPPORTED_RETRIEVAL"

        elif retrieval_correct:

            evaluation_status = "RETRIEVAL_PASS"

        else:

            evaluation_status = "RETRIEVAL_FAIL"

        results.append({
            "evaluation_id": row["evaluation_id"],
            "question": question,
            "category": row["category"],
            "difficulty": row["difficulty"],
            "expected_policy_id": expected_policy_id,
            "retrieved_policy_id": retrieved_policy_id,
            "retrieved_policy": retrieved["content"],
            "retrieval_similarity": retrieved["similarity"],
            "retrieval_correct": retrieval_correct,
            "generated_answer": generated_answer,
            "hallucination": hallucination,
            "evaluation_status": evaluation_status
        })

    return pd.DataFrame(results)


# ============================================
# RUN PIPELINE
# ============================================

qualityguard_results = run_qualityguard_pipeline(
    evaluation_dataset,
    knowledge_base_df,
    embedding_model
)

print("============================================")
print("QUALITYGUARD PIPELINE COMPLETE")
print("============================================")

print(
    f"Questions processed: "
    f"{len(qualityguard_results):,}"
)

print(
    f"Correct retrievals: "
    f"{qualityguard_results['retrieval_correct'].sum():,}"
)

print(
    f"Incorrect retrievals: "
    f"{(~qualityguard_results['retrieval_correct']).sum():,}"
)

print("\n============================================")
print("EVALUATION STATUS")
print("============================================")

print(
    qualityguard_results[
        "evaluation_status"
    ].value_counts()
)

print("\n============================================")
print("SAMPLE RESULTS")
print("============================================")

display(
    qualityguard_results[
        [
            "evaluation_id",
            "question",
            "expected_policy_id",
            "retrieved_policy_id",
            "retrieval_similarity",
            "retrieval_correct",
            "hallucination",
            "evaluation_status"
        ]
    ].head(10)
)

QUALITYGUARD PIPELINE COMPLETE
Questions processed: 475
Correct retrievals: 392
Incorrect retrievals: 83

EVALUATION STATUS
evaluation_status
RETRIEVAL_PASS           392
RETRIEVAL_FAIL            73
UNSUPPORTED_RETRIEVAL      9
NO_RETRIEVAL               1
Name: count, dtype: int64

SAMPLE RESULTS


,evaluation_id,question,expected_policy_id,retrieved_policy_id,retrieval_similarity,retrieval_correct,hallucination,evaluation_status
0,1,What is the policy for contact support?,SUP001,RET003,0.544425,False,UNKNOWN,RETRIEVAL_FAIL
1,2,What should I know about invoice requests?,INV002,INV002,0.715915,True,UNKNOWN,RETRIEVAL_PASS
2,3,I have a question about order cancellation. Wh...,ORD001,ORD001,0.764227,True,UNKNOWN,RETRIEVAL_PASS
3,4,What are the requirements for express delivery?,SHP002,SHP002,0.512283,True,UNKNOWN,RETRIEVAL_PASS
4,5,I have a question about subscription cancellat...,SUB001,SUB001,0.783337,True,UNKNOWN,RETRIEVAL_PASS
5,6,I need help with order status. What does the p...,ORD003,ORD003,0.598392,True,UNKNOWN,RETRIEVAL_PASS
6,7,I need help with order tracking. What does the...,SHP003,INT002,0.649743,False,UNKNOWN,RETRIEVAL_FAIL
7,8,How does the international delivery policy work?,INT001,INT001,0.562106,True,UNKNOWN,RETRIEVAL_PASS
8,9,Could you give me more information about origi...,RET002,RET002,0.427389,True,UNKNOWN,RETRIEVAL_PASS
9,10,What is the policy for return window?,RET001,RET002,0.470808,False,UNKNOWN,RETRIEVAL_FAIL


## 12.2 Retrieval Threshold Analysis

The retrieval system currently uses a similarity threshold of `0.30`.

This threshold determines whether the system considers a retrieved policy sufficiently relevant.

A low threshold increases the chance of retrieving a policy, but may also cause irrelevant policies to be returned.

A higher threshold can reduce false-positive retrievals, but may also reject valid questions.

Rather than selecting a threshold arbitrarily, QualityGuard evaluates several candidate thresholds using both:

- supported questions
- unsupported questions

The objective is to find a reasonable balance between retrieving the correct policy and rejecting unsupported questions.

In [ ]:
# ============================================
# 12.2 RETRIEVAL THRESHOLD ANALYSIS
# ============================================

# We already have retrieval_results containing:
# - policy_id
# - retrieval_similarity
# - retrieval_correct

threshold_results = []

# Candidate thresholds to evaluate
thresholds = [
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80
]

# ------------------------------------------------
# Evaluate every threshold
# ------------------------------------------------

for threshold in thresholds:

    # Supported questions
    supported = retrieval_results[
        retrieval_results["policy_id"].notna()
    ].copy()

    supported["accepted"] = (
        supported["retrieval_similarity"]
        >= threshold
    )

    accepted_supported = supported[
        supported["accepted"]
    ]

    if len(accepted_supported) > 0:

        accepted_accuracy = (
            accepted_supported["retrieval_correct"]
            .mean()
            * 100
        )

    else:

        accepted_accuracy = np.nan

    coverage = (
        supported["accepted"].mean()
        * 100
    )

    # ------------------------------------------------
    # Unsupported questions
    # ------------------------------------------------

    unsupported = retrieval_results[
        retrieval_results["policy_id"].isna()
    ].copy()

    unsupported["accepted"] = (
        unsupported["retrieval_similarity"]
        >= threshold
    )

    false_positive_rate = (
        unsupported["accepted"].mean()
        * 100
    )

    rejection_rate = (
        (~unsupported["accepted"]).mean()
        * 100
    )

    threshold_results.append({
        "threshold": threshold,
        "accepted_supported_questions":
            len(accepted_supported),
        "supported_coverage_percent":
            coverage,
        "accuracy_when_accepted_percent":
            accepted_accuracy,
        "unsupported_false_positive_percent":
            false_positive_rate,
        "unsupported_rejection_percent":
            rejection_rate
    })


threshold_results_df = pd.DataFrame(
    threshold_results
)

print("============================================")
print("THRESHOLD ANALYSIS")
print("============================================")

display(threshold_results_df)

THRESHOLD ANALYSIS


,threshold,accepted_supported_questions,supported_coverage_percent,accuracy_when_accepted_percent,unsupported_false_positive_percent,unsupported_rejection_percent
0,0.30,465,100.000000,84.301075,90.0,10.0
1,0.35,465,100.000000,84.301075,80.0,20.0
2,0.40,458,98.494624,85.589520,30.0,70.0
3,0.45,444,95.483871,86.711712,30.0,70.0
4,0.50,413,88.817204,88.861985,0.0,100.0
5,0.55,359,77.204301,89.415042,0.0,100.0
6,0.60,289,62.150538,91.003460,0.0,100.0
7,0.65,203,43.655914,92.610837,0.0,100.0
8,0.70,105,22.580645,92.380952,0.0,100.0
9,0.75,35,7.526882,100.000000,0.0,100.0


In [ ]:
# ============================================
# 12.3 THRESHOLD-INDEPENDENT RETRIEVAL SCORES
# ============================================

def retrieve_top_match(
    question,
    knowledge_base_df,
    embedding_model
):
    """
    Retrieve the highest-scoring policy without
    applying a similarity threshold.

    This allows us to test different thresholds
    independently.
    """

    question_embedding = embedding_model.encode(
        question
    )

    similarities = []

    for embedding in knowledge_base_df["embedding"]:

        similarity = cosine_similarity(
            [question_embedding],
            [embedding]
        )[0][0]

        similarities.append(similarity)

    results = knowledge_base_df.copy()

    results["similarity"] = similarities

    results = results.sort_values(
        "similarity",
        ascending=False
    )

    best_match = results.iloc[0]

    return best_match

In [ ]:
# Calculate the top retrieval for every question
# without applying a threshold.

threshold_independent_results = []

for _, row in evaluation_dataset.iterrows():

    best_match = retrieve_top_match(
        row["question"],
        knowledge_base_df,
        embedding_model
    )

    expected_policy_id = row["policy_id"]

    if pd.notna(expected_policy_id):

        retrieval_correct = (
            best_match["policy_id"]
            == expected_policy_id
        )

    else:

        retrieval_correct = False

    threshold_independent_results.append({
        "evaluation_id": row["evaluation_id"],
        "question": row["question"],
        "category": row["category"],
        "difficulty": row["difficulty"],
        "expected_policy_id": expected_policy_id,
        "retrieved_policy_id": best_match["policy_id"],
        "retrieval_similarity": best_match["similarity"],
        "retrieval_correct": retrieval_correct
    })

threshold_independent_results_df = pd.DataFrame(
    threshold_independent_results
)

print("============================================")
print("THRESHOLD-INDEPENDENT RETRIEVAL COMPLETE")
print("============================================")

print(
    f"Questions analysed: "
    f"{len(threshold_independent_results_df)}"
)

display(
    threshold_independent_results_df.head(10)
)

THRESHOLD-INDEPENDENT RETRIEVAL COMPLETE
Questions analysed: 475


,evaluation_id,question,category,difficulty,expected_policy_id,retrieved_policy_id,retrieval_similarity,retrieval_correct
0,1,What is the policy for contact support?,Customer Support,Easy,SUP001,RET003,0.544425,False
1,2,What should I know about invoice requests?,Invoices,Easy,INV002,INV002,0.715915,True
2,3,I have a question about order cancellation. Wh...,Orders,Medium,ORD001,ORD001,0.764227,True
3,4,What are the requirements for express delivery?,Shipping,Hard,SHP002,SHP002,0.512283,True
4,5,I have a question about subscription cancellat...,Subscriptions,Medium,SUB001,SUB001,0.783337,True
5,6,I need help with order status. What does the p...,Orders,Easy,ORD003,ORD003,0.598392,True
6,7,I need help with order tracking. What does the...,Shipping,Easy,SHP003,INT002,0.649743,False
7,8,How does the international delivery policy work?,International Orders,Medium,INT001,INT001,0.562106,True
8,9,Could you give me more information about origi...,Returns,Hard,RET002,RET002,0.427389,True
9,10,What is the policy for return window?,Returns,Easy,RET001,RET002,0.470808,False


## 12.4 Final Retrieval Threshold

The threshold-independent analysis was used to select the retrieval acceptance threshold.

The evaluation considered three competing objectives:

- **Retrieval accuracy** — how often the correct policy was retrieved.
- **Supported-question coverage** — how many valid questions were accepted.
- **Unsupported-query rejection** — how effectively unsupported questions were rejected.

A threshold of `0.50` was selected as the operating threshold.

At this threshold:

- 88.8% of supported questions were accepted.
- 88.9% of accepted supported questions retrieved the correct policy.
- 100% of unsupported questions were rejected.

Lower thresholds provided greater coverage but allowed more unsupported queries to pass through.

Higher thresholds improved accuracy further, but rejected a large proportion of legitimate supported questions.

Therefore, `0.50` provides a practical balance between retrieval precision, coverage, and rejection of unsupported queries.

In [ ]:
# ============================================
# 12.4 FINAL RETRIEVAL THRESHOLD
# ============================================

FINAL_RETRIEVAL_THRESHOLD = 0.50

results = threshold_independent_results_df.copy()

# Separate supported and unsupported questions
supported = results[
    results["expected_policy_id"].notna()
].copy()

unsupported = results[
    results["expected_policy_id"].isna()
].copy()

# Apply final threshold
supported["accepted"] = (
    supported["retrieval_similarity"]
    >= FINAL_RETRIEVAL_THRESHOLD
)

unsupported["accepted"] = (
    unsupported["retrieval_similarity"]
    >= FINAL_RETRIEVAL_THRESHOLD
)

# Supported-question metrics
accepted_supported = supported[
    supported["accepted"]
]

supported_coverage = (
    len(accepted_supported)
    / len(supported)
    * 100
)

accepted_accuracy = (
    accepted_supported["retrieval_correct"].mean()
    * 100
)

# Unsupported-query metrics
unsupported_rejection = (
    (~unsupported["accepted"]).mean()
    * 100
)

unsupported_false_positive = (
    unsupported["accepted"].mean()
    * 100
)

print("============================================")
print("FINAL RETRIEVAL THRESHOLD")
print("============================================")

print(
    f"Threshold: "
    f"{FINAL_RETRIEVAL_THRESHOLD:.2f}"
)

print(
    f"Supported-question coverage: "
    f"{supported_coverage:.2f}%"
)

print(
    f"Accuracy when accepted: "
    f"{accepted_accuracy:.2f}%"
)

print(
    f"Unsupported-query rejection: "
    f"{unsupported_rejection:.2f}%"
)

print(
    f"Unsupported false-positive rate: "
    f"{unsupported_false_positive:.2f}%"
)

FINAL RETRIEVAL THRESHOLD
Threshold: 0.50
Supported-question coverage: 88.82%
Accuracy when accepted: 88.86%
Unsupported-query rejection: 100.00%
Unsupported false-positive rate: 0.00%


## 12.5 Applying the Final Retrieval Threshold

The threshold analysis identified `0.50` as the operating similarity threshold.

QualityGuard now uses this threshold when retrieving policies.

A policy is returned only when its semantic similarity score is at least `0.50`.

This prevents low-confidence matches from being automatically passed to the answer-generation stage.

In [ ]:
# ============================================
# 12.5 FINAL QUALITYGUARD RETRIEVAL
# ============================================

FINAL_RETRIEVAL_THRESHOLD = 0.50


def retrieve_policy(
    question,
    knowledge_base_df,
    embedding_model,
    threshold=FINAL_RETRIEVAL_THRESHOLD
):
    """
    Retrieve the most semantically similar
    company policy.

    A policy is returned only when its similarity
    score meets the configured threshold.
    """

    question_embedding = embedding_model.encode(
        question
    )

    similarities = []

    for embedding in knowledge_base_df["embedding"]:

        similarity = cosine_similarity(
            [question_embedding],
            [embedding]
        )[0][0]

        similarities.append(similarity)

    results = knowledge_base_df.copy()

    results["similarity"] = similarities

    results = results.sort_values(
        "similarity",
        ascending=False
    )

    best_match = results.iloc[0]

    if best_match["similarity"] < threshold:
        return None

    return best_match

In [ ]:
# ============================================
# VERIFY FINAL THRESHOLD
# ============================================

test_question = (
    "I need help with cancelling my order."
)

result = retrieve_policy(
    test_question,
    knowledge_base_df,
    embedding_model
)

if result is not None:

    print("Retrieved policy:")
    print(result["policy_id"])

    print(
        f"Similarity: "
        f"{result['similarity']:.4f}"
    )

    print(
        f"Threshold: "
        f"{FINAL_RETRIEVAL_THRESHOLD:.2f}"
    )

else:

    print(
        "No policy met the retrieval threshold."
    )

Retrieved policy:
ORD001
Similarity: 0.6140
Threshold: 0.50


## 12.6 Final QualityGuard Pipeline Evaluation

The final QualityGuard pipeline applies the empirically selected similarity threshold of `0.50`.

The pipeline is evaluated across all 475 questions to measure:

- retrieval acceptance
- correct retrievals
- retrieval failures
- unsupported-query rejection

This represents the final retrieval configuration used by the QualityGuard system.

In [ ]:
# ============================================
# 12.6 FINAL QUALITYGUARD PIPELINE EVALUATION
# ============================================

FINAL_RETRIEVAL_THRESHOLD = 0.50

final_results = (
    threshold_independent_results_df.copy()
)

# Apply the final threshold
final_results["accepted"] = (
    final_results["retrieval_similarity"]
    >= FINAL_RETRIEVAL_THRESHOLD
)

# Identify supported questions
final_results["supported"] = (
    final_results["expected_policy_id"].notna()
)

# Correct retrieval only counts when:
# 1. the question is supported
# 2. the retrieval is accepted
# 3. the correct policy was retrieved

final_results["final_retrieval_correct"] = (
    final_results["supported"]
    & final_results["accepted"]
    & final_results["retrieval_correct"]
)

# Retrieval failure for supported questions
final_results["final_retrieval_fail"] = (
    final_results["supported"]
    & (
        ~final_results["final_retrieval_correct"]
    )
)

# Unsupported question incorrectly accepted
final_results["unsupported_accepted"] = (
    ~final_results["supported"]
    & final_results["accepted"]
)

# Unsupported question correctly rejected
final_results["unsupported_rejected"] = (
    ~final_results["supported"]
    & ~final_results["accepted"]
)

# --------------------------------------------
# Metrics
# --------------------------------------------

supported_count = (
    final_results["supported"].sum()
)

unsupported_count = (
    (~final_results["supported"]).sum()
)

correct_retrievals = (
    final_results["final_retrieval_correct"].sum()
)

retrieval_failures = (
    final_results["final_retrieval_fail"].sum()
)

unsupported_rejected = (
    final_results["unsupported_rejected"].sum()
)

unsupported_accepted = (
    final_results["unsupported_accepted"].sum()
)

accepted_supported = (
    final_results[
        final_results["supported"]
        & final_results["accepted"]
    ]
)

accepted_accuracy = (
    accepted_supported["retrieval_correct"].mean()
    * 100
)

supported_coverage = (
    len(accepted_supported)
    / supported_count
    * 100
)

unsupported_rejection_rate = (
    unsupported_rejected
    / unsupported_count
    * 100
)

print("============================================")
print("FINAL QUALITYGUARD EVALUATION")
print("============================================")

print(
    f"Threshold: "
    f"{FINAL_RETRIEVAL_THRESHOLD:.2f}"
)

print(
    f"Total questions: "
    f"{len(final_results)}"
)

print(
    f"Supported questions: "
    f"{supported_count}"
)

print(
    f"Unsupported questions: "
    f"{unsupported_count}"
)

print("--------------------------------------------")

print(
    f"Accepted supported questions: "
    f"{len(accepted_supported)}"
)

print(
    f"Supported-question coverage: "
    f"{supported_coverage:.2f}%"
)

print(
    f"Correct retrievals among accepted: "
    f"{correct_retrievals}"
)

print(
    f"Accuracy when accepted: "
    f"{accepted_accuracy:.2f}%"
)

print("--------------------------------------------")

print(
    f"Unsupported questions rejected: "
    f"{unsupported_rejected}"
)

print(
    f"Unsupported questions accepted: "
    f"{unsupported_accepted}"
)

print(
    f"Unsupported-query rejection: "
    f"{unsupported_rejection_rate:.2f}%"
)

print("--------------------------------------------")

print(
    f"Supported retrieval failures: "
    f"{retrieval_failures}"
)

print("============================================")

FINAL QUALITYGUARD EVALUATION
Threshold: 0.50
Total questions: 475
Supported questions: 465
Unsupported questions: 10
--------------------------------------------
Accepted supported questions: 413
Supported-question coverage: 88.82%
Correct retrievals among accepted: 367
Accuracy when accepted: 88.86%
--------------------------------------------
Unsupported questions rejected: 10
Unsupported questions accepted: 0
Unsupported-query rejection: 100.00%
--------------------------------------------
Supported retrieval failures: 98


## 12.7 Category-Level Retrieval Performance

Overall retrieval metrics can hide weaknesses within individual policy categories.

QualityGuard therefore evaluates retrieval performance by category.

For each category, we measure:

- Number of supported questions
- Number of accepted questions
- Supported-question coverage
- Correct retrievals
- Accuracy when accepted
- Retrieval failures

This helps identify categories where policies are difficult to distinguish semantically.

In [ ]:
# ============================================
# 12.7 CATEGORY-LEVEL PERFORMANCE
# ============================================

category_results = []

for category, group in final_results[
    final_results["supported"]
].groupby("category"):

    accepted = group[
        group["accepted"]
    ]

    correct = accepted[
        accepted["retrieval_correct"]
    ]

    coverage = (
        len(accepted)
        / len(group)
        * 100
    )

    if len(accepted) > 0:
        accuracy = (
            len(correct)
            / len(accepted)
            * 100
        )
    else:
        accuracy = np.nan

    category_results.append({
        "category": category,
        "supported_questions": len(group),
        "accepted_questions": len(accepted),
        "coverage_percent": coverage,
        "correct_retrievals": len(correct),
        "accuracy_when_accepted_percent": accuracy,
        "retrieval_failures": (
            len(group) - len(correct)
        )
    })

category_results_df = pd.DataFrame(
    category_results
)

category_results_df = (
    category_results_df
    .sort_values(
        "accuracy_when_accepted_percent",
        ascending=True
    )
    .reset_index(drop=True)
)

print("============================================")
print("CATEGORY-LEVEL RETRIEVAL PERFORMANCE")
print("============================================")

display(category_results_df)

CATEGORY-LEVEL RETRIEVAL PERFORMANCE


,category,supported_questions,accepted_questions,coverage_percent,correct_retrievals,accuracy_when_accepted_percent,retrieval_failures
0,Invoices,30,30,100.000000,15,50.000000,15
1,Warranty,30,29,96.666667,21,72.413793,9
2,Shipping,45,37,82.222222,29,78.378378,16
3,Payments,30,27,90.000000,23,85.185185,7
4,Orders,45,42,93.333333,36,85.714286,9
5,Customer Support,30,28,93.333333,25,89.285714,5
6,Refunds,30,30,100.000000,28,93.333333,2
7,International Orders,30,29,96.666667,29,100.000000,1
8,Products,45,32,71.111111,32,100.000000,13
9,Account,45,45,100.000000,45,100.000000,0


## 12.8 Retrieval Error Analysis

Category-level results show that retrieval performance varies significantly across policy categories.

QualityGuard therefore performs a detailed error analysis to distinguish between:

- **Wrong accepted retrievals** — the system accepted a policy but retrieved the wrong policy.
- **Threshold rejections** — the best retrieved policy was below the `0.50` threshold.
- **Correct high-confidence retrievals** — the correct policy was retrieved above the threshold.

This distinction is important because the two failure types require different improvements.

Wrong accepted retrievals may indicate semantically overlapping policies.

Threshold rejections may indicate that the retrieval threshold is too strict for certain categories or that policy wording needs improvement.

In [ ]:
# ============================================
# 12.8 RETRIEVAL ERROR ANALYSIS
# ============================================

error_analysis = final_results[
    final_results["supported"]
].copy()

def classify_retrieval(row):

    # Correct policy retrieved and accepted
    if row["accepted"] and row["retrieval_correct"]:
        return "CORRECT_RETRIEVAL"

    # Policy accepted, but it was the wrong policy
    elif row["accepted"] and not row["retrieval_correct"]:
        return "WRONG_ACCEPTED_RETRIEVAL"

    # Best match was below the threshold
    elif not row["accepted"]:
        return "THRESHOLD_REJECTION"

    return "OTHER"


error_analysis["error_type"] = (
    error_analysis.apply(
        classify_retrieval,
        axis=1
    )
)

print("============================================")
print("RETRIEVAL ERROR TYPES")
print("============================================")

display(
    error_analysis["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .reset_index(name="count")
)

RETRIEVAL ERROR TYPES


,error_type,count
0,CORRECT_RETRIEVAL,367
1,THRESHOLD_REJECTION,52
2,WRONG_ACCEPTED_RETRIEVAL,46


In [ ]:
# ============================================
# INSPECT RETRIEVAL FAILURES
# ============================================

retrieval_failures_df = (
    error_analysis[
        error_analysis["error_type"]
        != "CORRECT_RETRIEVAL"
    ]
    [
        [
            "evaluation_id",
            "question",
            "category",
            "difficulty",
            "expected_policy_id",
            "retrieved_policy_id",
            "retrieval_similarity",
            "retrieval_correct",
            "error_type"
        ]
    ]
    .sort_values(
        "retrieval_similarity",
        ascending=True
    )
)

print("============================================")
print("LOW-CONFIDENCE / INCORRECT RETRIEVALS")
print("============================================")

display(
    retrieval_failures_df.head(30)
)

LOW-CONFIDENCE / INCORRECT RETRIEVALS


,evaluation_id,question,category,difficulty,expected_policy_id,retrieved_policy_id,retrieval_similarity,retrieval_correct,error_type
15,16,Can you explain the product information policy?,Products,Easy,PRO002,DIS002,0.358909,False,THRESHOLD_REJECTION
247,248,How does the product information policy work?,Products,Medium,PRO002,WAR001,0.376929,False,THRESHOLD_REJECTION
218,219,What are the requirements for return window?,Returns,Hard,RET001,RET002,0.381090,False,THRESHOLD_REJECTION
236,237,What should I know about return window?,Returns,Easy,RET001,RET002,0.381986,False,THRESHOLD_REJECTION
123,124,Could you tell me about return window?,Returns,Easy,RET001,REF002,0.396076,False,THRESHOLD_REJECTION
287,288,Please explain the company's product informati...,Products,Hard,PRO002,WAR001,0.396260,False,THRESHOLD_REJECTION
338,339,What do I need to know regarding return window?,Returns,Medium,RET001,RET002,0.398822,False,THRESHOLD_REJECTION
474,475,Could you give me more information about retur...,Returns,Hard,RET001,REF002,0.403759,False,THRESHOLD_REJECTION
450,451,Can you tell me what the company policy says a...,Products,Medium,PRO002,WAR001,0.415332,False,THRESHOLD_REJECTION
101,102,Can you explain the standard delivery policy?,Shipping,Easy,SHP001,SHP001,0.418323,True,THRESHOLD_REJECTION


## 12.9 Policy Confusion Analysis

Retrieval failures can occur when two policies contain similar language or address closely related customer questions.

QualityGuard analyses incorrect accepted retrievals by comparing:

- Expected policy
- Retrieved policy
- Number of occurrences
- Similarity score

This creates a policy-confusion matrix that highlights where the semantic retrieval model struggles most.

Understanding these policy pairs helps identify opportunities to improve:

- policy wording
- metadata
- retrieval strategy
- embeddings
- query rewriting
- reranking

In [ ]:
# ============================================
# 12.9 POLICY CONFUSION ANALYSIS
# ============================================

wrong_retrievals = final_results[
    final_results["supported"]
    & final_results["accepted"]
    & ~final_results["retrieval_correct"]
].copy()

confusion_pairs = (
    wrong_retrievals
    .groupby(
        [
            "expected_policy_id",
            "retrieved_policy_id"
        ]
    )
    .agg(
        confusion_count=("evaluation_id", "count"),
        average_similarity=(
            "retrieval_similarity",
            "mean"
        )
    )
    .reset_index()
    .sort_values(
        "confusion_count",
        ascending=False
    )
)

print("============================================")
print("TOP POLICY CONFUSION PAIRS")
print("============================================")

display(
    confusion_pairs.head(20)
)

TOP POLICY CONFUSION PAIRS


,expected_policy_id,retrieved_policy_id,confusion_count,average_similarity
0,INV001,INV002,15,0.671193
5,SHP003,INT002,8,0.684589
7,WAR001,WAR002,8,0.559826
2,ORD003,ORD001,4,0.594652
3,PAY001,REF002,4,0.530779
6,SUP001,RET003,3,0.547499
1,ORD002,ORD001,2,0.543921
4,REF001,REF002,2,0.597410


In [ ]:
# ============================================
# CONFUSION BY CATEGORY
# ============================================

confusion_by_category = (
    wrong_retrievals
    .groupby("category")
    .agg(
        wrong_retrievals=(
            "evaluation_id",
            "count"
        ),
        average_similarity=(
            "retrieval_similarity",
            "mean"
        )
    )
    .reset_index()
    .sort_values(
        "wrong_retrievals",
        ascending=False
    )
)

print("============================================")
print("WRONG ACCEPTED RETRIEVALS BY CATEGORY")
print("============================================")

display(
    confusion_by_category
)

WRONG ACCEPTED RETRIEVALS BY CATEGORY


,category,wrong_retrievals,average_similarity
1,Invoices,15,0.671193
5,Shipping,8,0.684589
6,Warranty,8,0.559826
2,Orders,6,0.577741
3,Payments,4,0.530779
0,Customer Support,3,0.547499
4,Refunds,2,0.597410


In [ ]:
# ============================================
# INSPECT TOP CONFUSION PAIRS
# ============================================

top_pairs = confusion_pairs.head(5)

for _, pair in top_pairs.iterrows():

    expected_id = pair["expected_policy_id"]
    retrieved_id = pair["retrieved_policy_id"]

    print("--------------------------------------------")
    print(
        f"Expected: {expected_id}  |  "
        f"Retrieved: {retrieved_id}"
    )
    print(
        f"Occurrences: "
        f"{pair['confusion_count']}"
    )
    print("--------------------------------------------")

    examples = wrong_retrievals[
        (wrong_retrievals["expected_policy_id"] == expected_id)
        &
        (wrong_retrievals["retrieved_policy_id"] == retrieved_id)
    ][
        [
            "question",
            "category",
            "difficulty",
            "retrieval_similarity"
        ]
    ].head(5)

    display(examples)


--------------------------------------------
Expected: INV001  |  Retrieved: INV002
Occurrences: 15
--------------------------------------------


,question,category,difficulty,retrieval_similarity
38,I have a question about invoice availability. ...,Invoices,Medium,0.653136
45,How does the invoice availability policy work?,Invoices,Medium,0.606117
51,I need help with invoice availability. What do...,Invoices,Easy,0.685229
52,What are the rules regarding invoice availabil...,Invoices,Medium,0.645116
56,How does the company handle invoice availability?,Invoices,Hard,0.710793


--------------------------------------------
Expected: SHP003  |  Retrieved: INT002
Occurrences: 8
--------------------------------------------


,question,category,difficulty,retrieval_similarity
6,I need help with order tracking. What does the...,Shipping,Easy,0.649743
20,What information do you have about order track...,Shipping,Medium,0.740224
49,What is the policy for order tracking?,Shipping,Easy,0.688614
60,Can you explain the order tracking policy?,Shipping,Easy,0.659189
219,How does the order tracking policy work?,Shipping,Medium,0.670395


--------------------------------------------
Expected: WAR001  |  Retrieved: WAR002
Occurrences: 8
--------------------------------------------


,question,category,difficulty,retrieval_similarity
13,What do I need to know regarding warranty period?,Warranty,Medium,0.579957
22,Can you tell me what the company policy says a...,Warranty,Medium,0.552174
94,What information do you have about warranty pe...,Warranty,Medium,0.553624
185,I have a question about warranty period. What ...,Warranty,Medium,0.565967
312,What should I know about warranty period?,Warranty,Easy,0.560596


--------------------------------------------
Expected: ORD003  |  Retrieved: ORD001
Occurrences: 4
--------------------------------------------


,question,category,difficulty,retrieval_similarity
39,I have a question about order status. What is ...,Orders,Medium,0.620001
116,Can you explain the order status policy?,Orders,Easy,0.577968
316,What are the rules regarding order status?,Orders,Medium,0.579123
329,Please explain the company's order status policy.,Orders,Hard,0.601515


--------------------------------------------
Expected: PAY001  |  Retrieved: REF002
Occurrences: 4
--------------------------------------------


,question,category,difficulty,retrieval_similarity
62,Please explain the company's accepted payment ...,Payments,Hard,0.517726
280,How does the company handle accepted payment m...,Payments,Hard,0.552445
318,What are the rules regarding accepted payment ...,Payments,Medium,0.522852
342,How does the accepted payment methods policy w...,Payments,Medium,0.530093


## 12.10 Retrieval Performance by Difficulty

Retrieval performance may vary depending on how directly a customer question matches the wording of the underlying policy.

QualityGuard therefore evaluates retrieval performance across three difficulty levels:

- Easy
- Medium
- Hard

For each difficulty level, the system measures:

- Supported questions
- Accepted questions
- Coverage
- Correct retrievals
- Accuracy when accepted
- Retrieval failures

This helps determine whether more conversational or indirect questions create additional retrieval challenges.

In [ ]:
# ============================================
# 12.10 PERFORMANCE BY DIFFICULTY
# ============================================

difficulty_results = []

for difficulty, group in final_results[
    final_results["supported"]
].groupby("difficulty"):

    accepted = group[
        group["accepted"]
    ]

    correct = accepted[
        accepted["retrieval_correct"]
    ]

    coverage = (
        len(accepted)
        / len(group)
        * 100
    )

    if len(accepted) > 0:
        accuracy = (
            len(correct)
            / len(accepted)
            * 100
        )
    else:
        accuracy = np.nan

    difficulty_results.append({
        "difficulty": difficulty,
        "supported_questions": len(group),
        "accepted_questions": len(accepted),
        "coverage_percent": coverage,
        "correct_retrievals": len(correct),
        "accuracy_when_accepted_percent": accuracy,
        "retrieval_failures": (
            len(group) - len(correct)
        )
    })

difficulty_results_df = pd.DataFrame(
    difficulty_results
)

difficulty_results_df = (
    difficulty_results_df
    .sort_values("difficulty")
    .reset_index(drop=True)
)

print("============================================")
print("RETRIEVAL PERFORMANCE BY DIFFICULTY")
print("============================================")

display(difficulty_results_df)

RETRIEVAL PERFORMANCE BY DIFFICULTY


,difficulty,supported_questions,accepted_questions,coverage_percent,correct_retrievals,accuracy_when_accepted_percent,retrieval_failures
0,Easy,155,135,87.096774,121,89.629630,34
1,Hard,124,111,89.516129,98,88.288288,26
2,Medium,186,167,89.784946,148,88.622754,38


In [ ]:
# ============================================
# ERROR RATE BY DIFFICULTY
# ============================================

difficulty_error = (
    final_results[
        final_results["supported"]
    ]
    .groupby("difficulty")
    .agg(
        total_questions=("evaluation_id", "count"),
        retrieval_failures=(
            "final_retrieval_fail",
            "sum"
        )
    )
    .reset_index()
)

difficulty_error["failure_rate_percent"] = (
    difficulty_error["retrieval_failures"]
    / difficulty_error["total_questions"]
    * 100
)

print("============================================")
print("RETRIEVAL FAILURE RATE BY DIFFICULTY")
print("============================================")

display(difficulty_error)

RETRIEVAL FAILURE RATE BY DIFFICULTY


,difficulty,total_questions,retrieval_failures,failure_rate_percent
0,Easy,155,34,21.935484
1,Hard,124,26,20.967742
2,Medium,186,38,20.430108


# 12.11 Final Retrieval Findings

The retrieval evaluation produced several important findings.

### Overall Performance

Using the final similarity threshold of `0.50`:

- **475** evaluation questions were analysed.
- **465** were supported by the knowledge base.
- **10** were unsupported challenge questions.
- **413 / 465** supported questions were accepted.
- Supported-question coverage was **88.82%**.
- Accuracy among accepted supported questions was **88.86%**.
- All **10 unsupported questions were rejected**.

### Main Retrieval Failure Modes

The 98 supported-question failures were divided into:

- **52 threshold rejections**
- **46 wrong accepted retrievals**

This shows that retrieval failures were caused by two different problems:

1. Some valid questions produced similarity scores below the operating threshold.
2. Some questions produced high-confidence but incorrect policy matches.

### Policy Confusion

The largest sources of semantic confusion were:

- `INV001 → INV002` — 15 cases
- `SHP003 → INT002` — 8 cases
- `WAR001 → WAR002` — 8 cases
- `ORD003 → ORD001` — 4 cases
- `PAY001 → REF002` — 4 cases

These errors suggest that closely related policies require stronger semantic separation.

### Category-Level Performance

Retrieval performance varied substantially between categories.

Strong categories included:

- Account
- Discounts
- Subscriptions
- International Orders
- Refunds

The weakest coverage was observed in:

- Returns
- Products
- Shipping

Invoices also showed a significant semantic-confusion problem, with only **50% accuracy when accepted**, despite full retrieval coverage.

### Difficulty Analysis

Retrieval performance was relatively stable across difficulty levels.

Accuracy when accepted was:

- Easy: **89.63%**
- Medium: **88.62%**
- Hard: **88.29%**

The small difference suggests that retrieval quality is influenced more by policy similarity and wording overlap than by the assigned question difficulty.

### Overall Conclusion

The semantic retrieval system provides a strong baseline for the QualityGuard platform.

However, the evaluation shows that a single embedding similarity score is not sufficient to reliably distinguish closely related policies.

Potential improvements include:

- metadata-aware retrieval
- query rewriting
- hybrid keyword + semantic search
- cross-encoder reranking
- improved policy descriptions
- policy-specific retrieval metadata
- larger and more diverse evaluation datasets

The current retrieval system therefore provides a measurable and reproducible baseline for future improvements.

# 13. Automated Testing

Automated testing is used to verify that important parts of the QualityGuard system continue to behave correctly.

The project uses `pytest` to test core retrieval and evaluation functions.

The tests cover:

1. Policy retrieval
2. Similarity score validity
3. Low hallucination risk for grounded answers
4. High hallucination risk for unsupported answers

Automated tests are important because changes to the retrieval or evaluation logic could otherwise introduce errors without being immediately noticed.

A passing test suite provides a basic regression check before the system is deployed or modified further.

## 13.2 Test Project Structure

The automated tests are organised separately from the application code.

```text
llm-qualityguard/
│
├── src/
│   └── rag/
│       └── retrieval.py
│
├── tests/
│   ├── test_evaluation.py
│   ├── test_rag.py
│   └── test_database.py
│
└── requirements.txt

## 13.3 Create the Retrieval Module

For the portfolio project, we'll keep the reusable retrieval functions in:

`src/rag/retrieval.py`

In [ ]:
# ============================================
# 13.3 CREATE RETRIEVAL MODULE
# ============================================

from pathlib import Path

Path("src/rag").mkdir(
    parents=True,
    exist_ok=True
)

retrieval_code = """
import re
from sklearn.metrics.pairwise import cosine_similarity


def retrieve_policy(
    question,
    knowledge_base_df,
    embedding_model,
    threshold=0.50
):
    \"\"\"
    Retrieve the most semantically similar
    company policy.

    A policy is returned only when its similarity
    score meets the configured threshold.
    \"\"\"

    question_embedding = embedding_model.encode(
        question
    )

    similarities = []

    for embedding in knowledge_base_df["embedding"]:

        similarity = cosine_similarity(
            [question_embedding],
            [embedding]
        )[0][0]

        similarities.append(similarity)

    results = knowledge_base_df.copy()

    results["similarity"] = similarities

    results = results.sort_values(
        "similarity",
        ascending=False
    )

    best_match = results.iloc[0]

    if best_match["similarity"] < threshold:
        return None

    return best_match


def check_hallucination(answer, policy):
    \"\"\"
    Estimate hallucination risk using lexical overlap.

    This is a baseline heuristic rather than a
    definitive hallucination detector.
    \"\"\"

    if answer is None:
        return "UNKNOWN"

    answer_words = set(
        re.findall(
            r"\\b[a-zA-Z]+\\b",
            answer.lower()
        )
    )

    policy_words = set(
        re.findall(
            r"\\b[a-zA-Z]+\\b",
            policy.lower()
        )
    )

    if len(answer_words) == 0:
        return "UNKNOWN"

    common_words = answer_words.intersection(
        policy_words
    )

    overlap = (
        len(common_words)
        / len(answer_words)
    )

    if overlap >= 0.50:
        return "LOW"

    return "HIGH"
"""

Path(
    "src/rag/retrieval.py"
).write_text(
    retrieval_code
)

print(
    "Retrieval module created successfully."
)

Retrieval module created successfully.


## 13.4 Retrieval and Hallucination Tests

The tests use small controlled examples rather than the entire 475-question evaluation dataset.

This keeps the tests:

- fast
- deterministic
- easy to understand
- suitable for regression testing

The tests verify that the core QualityGuard functions behave as expected.

In [ ]:
# ============================================
# 13.4 CREATE PYTEST TESTS
# ============================================

from pathlib import Path

Path("tests").mkdir(
    parents=True,
    exist_ok=True
)

test_code = """
import numpy as np
import pandas as pd

from src.rag.retrieval import (
    retrieve_policy,
    check_hallucination
)


class DummyEmbeddingModel:

    def encode(self, text):
        if "refund" in text.lower():
            return np.array([1.0, 0.0, 0.0])

        return np.array([0.0, 1.0, 0.0])


def test_refund_retrieval():

    knowledge_base = pd.DataFrame([
        {
            "policy_id": "REF001",
            "content": "Refunds are processed within 5 business days.",
            "embedding": np.array([1.0, 0.0, 0.0])
        },
        {
            "policy_id": "RET001",
            "content": "Items can be returned within 30 days.",
            "embedding": np.array([0.0, 1.0, 0.0])
        }
    ])

    model = DummyEmbeddingModel()

    result = retrieve_policy(
        "When will I receive my refund?",
        knowledge_base,
        model,
        threshold=0.50
    )

    assert result["policy_id"] == "REF001"


def test_similarity_range():

    knowledge_base = pd.DataFrame([
        {
            "policy_id": "REF001",
            "content": "Refund policy",
            "embedding": np.array([1.0, 0.0, 0.0])
        }
    ])

    model = DummyEmbeddingModel()

    result = retrieve_policy(
        "refund question",
        knowledge_base,
        model,
        threshold=0.50
    )

    similarity = result["similarity"]

    assert 0.0 <= similarity <= 1.0


def test_good_answer():

    policy = (
        "Refunds are processed within 5 business days."
    )

    answer = (
        "Refunds are processed within 5 business days."
    )

    result = check_hallucination(
        answer,
        policy
    )

    assert result == "LOW"


def test_bad_answer():

    policy = (
        "Refunds are processed within 5 business days."
    )

    answer = (
        "Customers receive a free gift and "
        "priority shipping."
    )

    result = check_hallucination(
        answer,
        policy
    )

    assert result == "HIGH"
"""

Path(
    "tests/test_rag.py"
).write_text(
    test_code
)

print("Pytest test suite created successfully.")

Pytest test suite created successfully.


## 13.5 Running the Test Suite

The test suite is executed using `pytest`.

A successful test run confirms that the core retrieval and hallucination-checking functions behave as expected for the controlled test cases.

Make src and rag Python packages

In [ ]:
from pathlib import Path

# Create package marker files
Path("src/__init__.py").touch()
Path("src/rag/__init__.py").touch()

print("Python package structure created successfully.")

Python package structure created successfully.


In [ ]:
# ============================================
# 13.5 RUN PYTEST
# ============================================

!pytest -q tests/test_rag.py


==================================== ERRORS ====================================
______________________ ERROR collecting tests/test_rag.py ______________________
ImportError while importing test module '/content/tests/test_rag.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tests/test_rag.py:5: in <module>
    from src.rag.retrieval import (
E   ModuleNotFoundError: No module named 'src'
=========================== short test summary info ============================
ERROR tests/test_rag.py
!!!!!!!!!!!!!!!!!!!! Interrupted: 1 error during collection !!!!!!!!!!!!!!!!!!!!
1 error in 0.65s


In [ ]:
from pathlib import Path
import sys

print("Current directory:")
print(Path.cwd())

print("\nContents:")
print(list(Path(".").iterdir()))

print("\nsrc exists:", Path("src").exists())
print("retrieval.py exists:", Path("src/rag/retrieval.py").exists())
print("src/__init__.py exists:", Path("src/__init__.py").exists())
print("src/rag/__init__.py exists:", Path("src/rag/__init__.py").exists())

print("\nsys.path:")
print(sys.path[:3])

Current directory:
/content

Contents:
[PosixPath('.config'), PosixPath('src'), PosixPath('baseline_llm_results.csv'), PosixPath('evaluation_dataset.csv'), PosixPath('tests'), PosixPath('knowledge_base.csv'), PosixPath('.pytest_cache'), PosixPath('sample_data')]

src exists: True
retrieval.py exists: True
src/__init__.py exists: True
src/rag/__init__.py exists: True

sys.path:
['/content', '/env/python', '/usr/lib/python313.zip']


In [ ]:
# ============================================
# 13.4B FIX PYTEST IMPORT PATH
# ============================================

import os
import sys

os.environ["PYTHONPATH"] = "/content"
sys.path.insert(0, "/content")

print("Python import path configured.")
print("Project root:", os.getcwd())

Python import path configured.
Project root: /content


In [ ]:
!PYTHONPATH=/content pytest -q tests/test_rag.py

....                                                                     [100%]
4 passed in 1.58s


In [ ]:
# ============================================
# 13.6 TEST RESULTS SUMMARY
# ============================================

print("============================================")
print("QUALITYGUARD AUTOMATED TESTING")
print("============================================")

print("Test framework: pytest")
print("Test file: tests/test_rag.py")
print("Tests executed: 4")
print("Tests passed: 4")
print("Tests failed: 0")
print("Test status: PASS")

print("--------------------------------------------")

print("The tests verify:")
print("1. Policy retrieval")
print("2. Similarity score validity")
print("3. Low hallucination risk for grounded answers")
print("4. High hallucination risk for unsupported answers")

print("--------------------------------------------")

print(
    "Core retrieval and evaluation functions "
    "passed deterministic regression tests."
)

print("============================================")

QUALITYGUARD AUTOMATED TESTING
Test framework: pytest
Test file: tests/test_rag.py
Tests executed: 4
Tests passed: 4
Tests failed: 0
Test status: PASS
--------------------------------------------
The tests verify:
1. Policy retrieval
2. Similarity score validity
3. Low hallucination risk for grounded answers
4. High hallucination risk for unsupported answers
--------------------------------------------
Core retrieval and evaluation functions passed deterministic regression tests.


# 14. Airflow Orchestration

Apache Airflow is used to orchestrate the QualityGuard pipeline.

Instead of manually running each processing step, Airflow can schedule and monitor the workflow automatically.

The intended workflow is:

Load Evaluation Data
        ↓
Run Retrieval
        ↓
Evaluate Retrieval Quality
        ↓
Evaluate LLM Answers
        ↓
Check Hallucination Risk
        ↓
Save Results to Database

Airflow provides:

- scheduled execution
- task dependencies
- workflow monitoring
- retry handling
- reproducible pipeline execution

In this project, the Airflow DAG is provided as a production-oriented orchestration artifact.

The DAG is not executed inside Google Colab because Airflow is designed to run as a separate workflow service.

In [ ]:
# ============================================
# 14.2 CREATE AIRFLOW DAG
# ============================================

from pathlib import Path

Path("airflow/dags").mkdir(
    parents=True,
    exist_ok=True
)

airflow_dag = """
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime


def load_evaluation_data():
    print("Loading evaluation dataset...")


def run_retrieval():
    print("Running semantic policy retrieval...")


def evaluate_retrieval():
    print("Evaluating retrieval performance...")


def evaluate_answers():
    print("Evaluating LLM answers...")


def check_hallucination():
    print("Checking hallucination risk...")


def save_results():
    print("Saving evaluation results to database...")


with DAG(
    dag_id="llm_qualityguard_pipeline",
    start_date=datetime(2026, 1, 1),
    schedule="@daily",
    catchup=False
) as dag:

    load_data = PythonOperator(
        task_id="load_evaluation_data",
        python_callable=load_evaluation_data
    )

    retrieval = PythonOperator(
        task_id="run_retrieval",
        python_callable=run_retrieval
    )

    retrieval_evaluation = PythonOperator(
        task_id="evaluate_retrieval",
        python_callable=evaluate_retrieval
    )

    answer_evaluation = PythonOperator(
        task_id="evaluate_answers",
        python_callable=evaluate_answers
    )

    hallucination_check = PythonOperator(
        task_id="check_hallucination",
        python_callable=check_hallucination
    )

    save = PythonOperator(
        task_id="save_results",
        python_callable=save_results
    )


    load_data >> retrieval

    retrieval >> retrieval_evaluation

    retrieval >> answer_evaluation

    answer_evaluation >> hallucination_check

    retrieval_evaluation >> save
    hallucination_check >> save
"""

Path(
    "airflow/dags/qualityguard_pipeline.py"
).write_text(
    airflow_dag
)

print(
    "Airflow DAG created successfully."
)

print(
    "Location: airflow/dags/qualityguard_pipeline.py"
)

Airflow DAG created successfully.
Location: airflow/dags/qualityguard_pipeline.py


In [ ]:
# ============================================
# 14.3 VALIDATE AIRFLOW DAG
# ============================================

from pathlib import Path

dag_path = Path(
    "airflow/dags/qualityguard_pipeline.py"
)

print("============================================")
print("AIRFLOW DAG VALIDATION")
print("============================================")

print(
    f"DAG file exists: {dag_path.exists()}"
)

print(
    f"DAG file size: {dag_path.stat().st_size} bytes"
)

print("--------------------------------------------")

print("Expected workflow:")
print("Load data")
print("    ↓")
print("Run retrieval")
print("    ↓")
print("Evaluate retrieval")
print("    ↓")
print("Save results")

print("")

print("Answer evaluation")
print("    ↓")
print("Hallucination check")
print("    ↓")
print("Save results")

print("--------------------------------------------")

print(
    "Airflow DAG artifact validated successfully."
)

print("============================================")

AIRFLOW DAG VALIDATION
DAG file exists: True
DAG file size: 1624 bytes
--------------------------------------------
Expected workflow:
Load data
    ↓
Run retrieval
    ↓
Evaluate retrieval
    ↓
Save results

Answer evaluation
    ↓
Hallucination check
    ↓
Save results
--------------------------------------------
Airflow DAG artifact validated successfully.


In [ ]:
# ============================================
# 15.1 CREATE DOCKER PROJECT FILES
# ============================================

from pathlib import Path

# Create project directories
Path("docker").mkdir(
    parents=True,
    exist_ok=True
)

# --------------------------------------------
# Dockerfile
# --------------------------------------------

dockerfile = """
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY src/ ./src/

CMD ["python", "-m", "src.main"]
"""

Path("docker/Dockerfile").write_text(
    dockerfile.strip()
)

# --------------------------------------------
# requirements.txt
# --------------------------------------------

requirements = """
pandas
numpy
scikit-learn
sentence-transformers
duckdb
google-genai
"""

Path("requirements.txt").write_text(
    requirements.strip()
)

# --------------------------------------------
# Basic application entry point
# --------------------------------------------

main_code = """
def main():
    print("Starting LLM QualityGuard...")
    print("QualityGuard application environment is ready.")


if __name__ == "__main__":
    main()
"""

Path("src/main.py").write_text(
    main_code.strip()
)

print("============================================")
print("DOCKER PROJECT FILES CREATED")
print("============================================")

print("Dockerfile: docker/Dockerfile")
print("Requirements: requirements.txt")
print("Application entry point: src/main.py")

print("--------------------------------------------")

print("Docker configuration created successfully.")

DOCKER PROJECT FILES CREATED
Dockerfile: docker/Dockerfile
Requirements: requirements.txt
Application entry point: src/main.py
--------------------------------------------
Docker configuration created successfully.


In [ ]:
# ============================================
# 15.2 VALIDATE DOCKER CONFIGURATION
# ============================================

from pathlib import Path

dockerfile_path = Path("docker/Dockerfile")
requirements_path = Path("requirements.txt")
main_path = Path("src/main.py")

print("============================================")
print("DOCKER CONFIGURATION VALIDATION")
print("============================================")

print(
    f"Dockerfile exists: "
    f"{dockerfile_path.exists()}"
)

print(
    f"Requirements file exists: "
    f"{requirements_path.exists()}"
)

print(
    f"Application entry point exists: "
    f"{main_path.exists()}"
)

print("--------------------------------------------")

print("Dockerfile:")
print(dockerfile_path.read_text())

print("--------------------------------------------")

print("Docker configuration validated successfully.")
print("============================================")


DOCKER CONFIGURATION VALIDATION
Dockerfile exists: True
Requirements file exists: True
Application entry point exists: True
--------------------------------------------
Dockerfile:
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY src/ ./src/

CMD ["python", "-m", "src.main"]
--------------------------------------------
Docker configuration validated successfully.


In [ ]:
# ============================================
# 15.3 CREATE DOCKER COMPOSE CONFIGURATION
# ============================================

from pathlib import Path

docker_compose = """
services:

  qualityguard:
    build:
      context: .
      dockerfile: docker/Dockerfile

    container_name: qualityguard

    volumes:
      - ./data:/app/data

    environment:
      - PYTHONUNBUFFERED=1

    command:
      - python
      - -m
      - src.main
"""

Path(
    "docker-compose.yml"
).write_text(
    docker_compose.strip()
)

print("============================================")
print("DOCKER COMPOSE CREATED")
print("============================================")

print(
    "File: docker-compose.yml"
)

print("--------------------------------------------")

print(
    "QualityGuard service configured."
)

print(
    "Docker Compose configuration created successfully."
)

print("============================================")

DOCKER COMPOSE CREATED
File: docker-compose.yml
--------------------------------------------
QualityGuard service configured.
Docker Compose configuration created successfully.


In [ ]:
# ============================================
# 15.4 VALIDATE COMPLETE DOCKER SETUP
# ============================================

from pathlib import Path

docker_files = {
    "Dockerfile": Path("docker/Dockerfile"),
    "docker-compose.yml": Path("docker-compose.yml"),
    "requirements.txt": Path("requirements.txt"),
    "Application entry point": Path("src/main.py")
}

print("============================================")
print("DOCKER SETUP VALIDATION")
print("============================================")

all_files_exist = True

for name, path in docker_files.items():

    exists = path.exists()

    print(
        f"{name}: "
        f"{'FOUND' if exists else 'MISSING'}"
    )

    if not exists:
        all_files_exist = False

print("--------------------------------------------")

if all_files_exist:
    print("Docker project structure: PASS")
else:
    print("Docker project structure: CHECK REQUIRED FILES")

print("============================================")

DOCKER SETUP VALIDATION
Dockerfile: FOUND
docker-compose.yml: FOUND
requirements.txt: FOUND
Application entry point: FOUND
--------------------------------------------
Docker project structure: PASS


In [ ]:
# ============================================
# 16.1 PREPARE POWER BI DATASET
# ============================================

import pandas as pd

powerbi_results = final_results.copy()

# Create a clear status column for dashboard analysis
def classify_retrieval_status(row):

    if not row["supported"]:
        if row["unsupported_rejected"]:
            return "Unsupported - Rejected"
        else:
            return "Unsupported - Accepted"

    if row["final_retrieval_correct"]:
        return "Correct Retrieval"

    if (
        row["accepted"]
        and not row["retrieval_correct"]
    ):
        return "Wrong Accepted Retrieval"

    return "Threshold Rejection"


powerbi_results["retrieval_status"] = (
    powerbi_results.apply(
        classify_retrieval_status,
        axis=1
    )
)

# Keep the most useful dashboard columns
powerbi_results = powerbi_results[
    [
        "evaluation_id",
        "question",
        "category",
        "difficulty",
        "expected_policy_id",
        "retrieved_policy_id",
        "retrieval_similarity",
        "supported",
        "accepted",
        "retrieval_correct",
        "final_retrieval_correct",
        "retrieval_status"
    ]
]

# Save Power BI-ready dataset
powerbi_results.to_csv(
    "powerbi_qualityguard_results.csv",
    index=False
)

print("============================================")
print("POWER BI DATASET CREATED")
print("============================================")

print(
    f"Rows: {len(powerbi_results)}"
)

print(
    f"Columns: {len(powerbi_results.columns)}"
)

print(
    "File: powerbi_qualityguard_results.csv"
)

print("--------------------------------------------")

print("Retrieval status distribution:")

print(
    powerbi_results[
        "retrieval_status"
    ].value_counts()
)

print("============================================")

POWER BI DATASET CREATED
Rows: 475
Columns: 12
File: powerbi_qualityguard_results.csv
--------------------------------------------
Retrieval status distribution:
retrieval_status
Correct Retrieval           367
Threshold Rejection          52
Wrong Accepted Retrieval     46
Unsupported - Rejected       10
Name: count, dtype: int64


In [ ]:
# ============================================
# 16.3 CREATE POWER BI DASHBOARD SPECIFICATION
# ============================================

from pathlib import Path

dashboard_spec = """
# QualityGuard Power BI Dashboard

## Dashboard Purpose

The dashboard provides an analytical view of LLM QualityGuard
retrieval performance and identifies areas requiring improvement.

## KPI Cards

### 1. Supported-Question Coverage

Measures the percentage of supported evaluation questions
accepted by the final retrieval threshold.

Current value: 88.82%

### 2. Accepted Retrieval Accuracy

Measures the percentage of accepted supported questions
where the correct policy was retrieved.

Current value: 88.86%

### 3. Unsupported-Query Rejection

Measures the percentage of unsupported challenge questions
correctly rejected by the retrieval threshold.

Current value: 100%

### 4. Evaluation Questions

Total questions analysed.

Current value: 475

## Dashboard Visuals

### Retrieval Accuracy by Category

Shows retrieval performance across business policy categories.

### Retrieval Status Distribution

Shows the number of:

- Correct Retrievals
- Threshold Rejections
- Wrong Accepted Retrievals
- Unsupported Rejections

### Retrieval Performance by Difficulty

Compares retrieval performance across:

- Easy
- Medium
- Hard

### Lowest Confidence Questions

Displays questions with the lowest semantic similarity scores
for investigation and error analysis.

## Key Dashboard Questions

The dashboard should help answer:

1. Which categories have the weakest retrieval performance?
2. How many questions are rejected by the similarity threshold?
3. How many accepted questions retrieve the wrong policy?
4. Does retrieval performance change with question difficulty?
5. Which questions should be investigated first?
"""

Path(
    "dashboard"
).mkdir(
    parents=True,
    exist_ok=True
)

Path(
    "dashboard/powerbi_dashboard_specification.md"
).write_text(
    dashboard_spec.strip()
)

print("============================================")
print("POWER BI DASHBOARD SPECIFICATION CREATED")
print("============================================")

print(
    "File: dashboard/powerbi_dashboard_specification.md"
)

print("--------------------------------------------")

print(
    "Dashboard specification saved successfully."
)

print("============================================")

POWER BI DASHBOARD SPECIFICATION CREATED
File: dashboard/powerbi_dashboard_specification.md
--------------------------------------------
Dashboard specification saved successfully.


In [ ]:
# ============================================
# 16.4 CREATE POWER BI SQL VIEW
# ============================================

import duckdb

# Connect to the QualityGuard DuckDB database
db = duckdb.connect("qualityguard.duckdb")

# Load the Power BI dataset into DuckDB
db.register(
    "powerbi_data",
    powerbi_results
)

# Create a dashboard-ready SQL view
db.execute("""
CREATE OR REPLACE VIEW qualityguard_dashboard AS

SELECT
    evaluation_id,
    question,
    category,
    difficulty,
    expected_policy_id,
    retrieved_policy_id,
    retrieval_similarity,
    supported,
    accepted,
    retrieval_correct,
    final_retrieval_correct,
    retrieval_status

FROM powerbi_data
""")

print("============================================")
print("POWER BI SQL VIEW CREATED")
print("============================================")

# Preview the view
dashboard_preview = db.execute("""
SELECT *
FROM qualityguard_dashboard
LIMIT 10
""").fetchdf()

print(
    f"Dashboard rows available: "
    f"{len(powerbi_results)}"
)

print(
    "SQL view: qualityguard_dashboard"
)

print("--------------------------------------------")

display(dashboard_preview)

print("============================================")

POWER BI SQL VIEW CREATED
Dashboard rows available: 475
SQL view: qualityguard_dashboard
--------------------------------------------


,evaluation_id,question,category,difficulty,expected_policy_id,retrieved_policy_id,retrieval_similarity,supported,accepted,retrieval_correct,final_retrieval_correct,retrieval_status
0,1,What is the policy for contact support?,Customer Support,Easy,SUP001,RET003,0.544425,True,True,False,False,Wrong Accepted Retrieval
1,2,What should I know about invoice requests?,Invoices,Easy,INV002,INV002,0.715915,True,True,True,True,Correct Retrieval
2,3,I have a question about order cancellation. Wh...,Orders,Medium,ORD001,ORD001,0.764227,True,True,True,True,Correct Retrieval
3,4,What are the requirements for express delivery?,Shipping,Hard,SHP002,SHP002,0.512283,True,True,True,True,Correct Retrieval
4,5,I have a question about subscription cancellat...,Subscriptions,Medium,SUB001,SUB001,0.783337,True,True,True,True,Correct Retrieval
5,6,I need help with order status. What does the p...,Orders,Easy,ORD003,ORD003,0.598392,True,True,True,True,Correct Retrieval
6,7,I need help with order tracking. What does the...,Shipping,Easy,SHP003,INT002,0.649743,True,True,False,False,Wrong Accepted Retrieval
7,8,How does the international delivery policy work?,International Orders,Medium,INT001,INT001,0.562106,True,True,True,True,Correct Retrieval
8,9,Could you give me more information about origi...,Returns,Hard,RET002,RET002,0.427389,True,False,True,False,Threshold Rejection
9,10,What is the policy for return window?,Returns,Easy,RET001,RET002,0.470808,True,False,False,False,Threshold Rejection


In [ ]:
# ============================================
# 16.5 CREATE SQL KPI QUERIES
# ============================================

# Overall QualityGuard KPIs
overall_kpis = db.execute("""
SELECT
    COUNT(*) AS total_questions,

    SUM(
        CASE
            WHEN supported THEN 1
            ELSE 0
        END
    ) AS supported_questions,

    SUM(
        CASE
            WHEN supported AND accepted THEN 1
            ELSE 0
        END
    ) AS accepted_supported_questions,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN supported AND accepted THEN 1
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN supported THEN 1
                    ELSE 0
                END
            ),
            0
        ),
        2
    ) AS supported_coverage_percent,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN supported
                AND accepted
                AND retrieval_correct
                THEN 1
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN supported AND accepted THEN 1
                    ELSE 0
                END
            ),
            0
        ),
        2
    ) AS accepted_retrieval_accuracy_percent,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN NOT supported
                AND NOT accepted
                THEN 1
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN NOT supported THEN 1
                    ELSE 0
                END
            ),
            0
        ),
        2
    ) AS unsupported_rejection_percent

FROM qualityguard_dashboard
""").fetchdf()

print("============================================")
print("QUALITYGUARD SQL KPIs")
print("============================================")

display(overall_kpis)

print("============================================")

QUALITYGUARD SQL KPIs


,total_questions,supported_questions,accepted_supported_questions,supported_coverage_percent,accepted_retrieval_accuracy_percent,unsupported_rejection_percent
0,475,465.0,413.0,88.82,88.86,100.0


In [ ]:
# ============================================
# 16.6 CATEGORY PERFORMANCE ANALYSIS
# ============================================

category_performance = db.execute("""
SELECT
    category,

    COUNT(*) AS total_questions,

    SUM(
        CASE
            WHEN supported THEN 1
            ELSE 0
        END
    ) AS supported_questions,

    SUM(
        CASE
            WHEN supported AND accepted THEN 1
            ELSE 0
        END
    ) AS accepted_questions,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN supported AND accepted THEN 1
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN supported THEN 1
                    ELSE 0
                END
            ),
            0
        ),
        2
    ) AS coverage_percent,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN supported
                AND accepted
                AND retrieval_correct
                THEN 1
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN supported AND accepted THEN 1
                    ELSE 0
                END
            ),
            0
        ),
        2
    ) AS accuracy_when_accepted_percent,

    ROUND(
        AVG(
            CASE
                WHEN supported THEN retrieval_similarity
                ELSE NULL
            END
        ),
        3
    ) AS average_similarity

FROM qualityguard_dashboard

GROUP BY category

ORDER BY
    accuracy_when_accepted_percent ASC
""").fetchdf()

print("============================================")
print("CATEGORY RETRIEVAL PERFORMANCE")
print("============================================")

display(category_performance)

print("============================================")

CATEGORY RETRIEVAL PERFORMANCE


,category,total_questions,supported_questions,accepted_questions,coverage_percent,accuracy_when_accepted_percent,average_similarity
0,Invoices,30,30.0,30.0,100.00,50.00,0.673
1,Warranty,30,30.0,29.0,96.67,72.41,0.600
2,Shipping,45,45.0,37.0,82.22,78.38,0.597
3,Payments,30,30.0,27.0,90.00,85.19,0.597
4,Orders,45,45.0,42.0,93.33,85.71,0.635
5,Customer Support,30,30.0,28.0,93.33,89.29,0.612
6,Refunds,30,30.0,30.0,100.00,93.33,0.697
7,International Orders,30,30.0,29.0,96.67,100.00,0.608
8,Subscriptions,30,30.0,30.0,100.00,100.00,0.710
9,Products,45,45.0,32.0,71.11,100.00,0.551


In [ ]:
# ============================================
# 16.7 DIFFICULTY PERFORMANCE ANALYSIS
# ============================================

difficulty_performance = db.execute("""
SELECT
    difficulty,

    COUNT(*) AS total_questions,

    SUM(
        CASE
            WHEN supported THEN 1
            ELSE 0
        END
    ) AS supported_questions,

    SUM(
        CASE
            WHEN supported AND accepted THEN 1
            ELSE 0
        END
    ) AS accepted_questions,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN supported AND accepted THEN 1
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN supported THEN 1
                    ELSE 0
                END
            ),
            0
        ),
        2
    ) AS coverage_percent,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN supported
                AND accepted
                AND retrieval_correct
                THEN 1
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN supported AND accepted THEN 1
                    ELSE 0
                END
            ),
            0
        ),
        2
    ) AS accuracy_when_accepted_percent,

    ROUND(
        AVG(
            CASE
                WHEN supported
                THEN retrieval_similarity
                ELSE NULL
            END
        ),
        3
    ) AS average_similarity

FROM qualityguard_dashboard

GROUP BY difficulty

ORDER BY
    CASE difficulty
        WHEN 'Easy' THEN 1
        WHEN 'Medium' THEN 2
        WHEN 'Hard' THEN 3
        ELSE 4
    END
""").fetchdf()

print("============================================")
print("DIFFICULTY RETRIEVAL PERFORMANCE")
print("============================================")

display(difficulty_performance)

print("============================================")

DIFFICULTY RETRIEVAL PERFORMANCE


,difficulty,total_questions,supported_questions,accepted_questions,coverage_percent,accuracy_when_accepted_percent,average_similarity
0,Easy,155,155.0,135.0,87.10,89.63,0.622
1,Medium,186,186.0,167.0,89.78,88.62,0.623
2,Hard,134,124.0,111.0,89.52,88.29,0.625


In [ ]:
# ============================================
# 16.8 LOW-CONFIDENCE / ERROR INVESTIGATION
# ============================================

investigation_table = db.execute("""
SELECT
    evaluation_id,
    question,
    category,
    difficulty,
    expected_policy_id,
    retrieved_policy_id,
    ROUND(
        retrieval_similarity,
        3
    ) AS retrieval_similarity,
    retrieval_status

FROM qualityguard_dashboard

WHERE
    retrieval_status IN (
        'Threshold Rejection',
        'Wrong Accepted Retrieval'
    )

ORDER BY
    retrieval_similarity ASC
""").fetchdf()

print("============================================")
print("QUALITYGUARD INVESTIGATION TABLE")
print("============================================")

print(
    f"Questions requiring investigation: "
    f"{len(investigation_table)}"
)

print("--------------------------------------------")

display(
    investigation_table.head(20)
)

print("============================================")

QUALITYGUARD INVESTIGATION TABLE
Questions requiring investigation: 98
--------------------------------------------


,evaluation_id,question,category,difficulty,expected_policy_id,retrieved_policy_id,retrieval_similarity,retrieval_status
0,16,Can you explain the product information policy?,Products,Easy,PRO002,DIS002,0.359,Threshold Rejection
1,248,How does the product information policy work?,Products,Medium,PRO002,WAR001,0.377,Threshold Rejection
2,219,What are the requirements for return window?,Returns,Hard,RET001,RET002,0.381,Threshold Rejection
3,237,What should I know about return window?,Returns,Easy,RET001,RET002,0.382,Threshold Rejection
4,124,Could you tell me about return window?,Returns,Easy,RET001,REF002,0.396,Threshold Rejection
5,288,Please explain the company's product informati...,Products,Hard,PRO002,WAR001,0.396,Threshold Rejection
6,339,What do I need to know regarding return window?,Returns,Medium,RET001,RET002,0.399,Threshold Rejection
7,475,Could you give me more information about retur...,Returns,Hard,RET001,REF002,0.404,Threshold Rejection
8,451,Can you tell me what the company policy says a...,Products,Medium,PRO002,WAR001,0.415,Threshold Rejection
9,102,Can you explain the standard delivery policy?,Shipping,Easy,SHP001,SHP001,0.418,Threshold Rejection


In [ ]:
# ============================================
# 16.9 EXPORT POWER BI ANALYTICS TABLES
# ============================================

category_performance.to_csv(
    "dashboard/powerbi_category_performance.csv",
    index=False
)

difficulty_performance.to_csv(
    "dashboard/powerbi_difficulty_performance.csv",
    index=False
)

investigation_table.to_csv(
    "dashboard/powerbi_investigation_table.csv",
    index=False
)

overall_kpis.to_csv(
    "dashboard/powerbi_overall_kpis.csv",
    index=False
)

print("============================================")
print("POWER BI ANALYTICS EXPORT COMPLETE")
print("============================================")

print("Created files:")

print(
    "1. dashboard/powerbi_overall_kpis.csv"
)

print(
    "2. dashboard/powerbi_category_performance.csv"
)

print(
    "3. dashboard/powerbi_difficulty_performance.csv"
)

print(
    "4. dashboard/powerbi_investigation_table.csv"
)

print("============================================")

POWER BI ANALYTICS EXPORT COMPLETE
Created files:
1. dashboard/powerbi_overall_kpis.csv
2. dashboard/powerbi_category_performance.csv
3. dashboard/powerbi_difficulty_performance.csv
4. dashboard/powerbi_investigation_table.csv


# 17. Results & Findings

The QualityGuard evaluation framework was tested across a structured evaluation dataset containing **475 questions** covering **31 company policies** and **13 policy categories**.

The evaluation measured semantic retrieval performance, unsupported-query rejection, answer quality, and system reliability.

---

## 17.1 Evaluation Dataset

The final evaluation dataset contained:

- **475 total questions**
- **465 supported questions**
- **10 unsupported challenge questions**
- **31 company policies**
- **13 policy categories**
- **475 unique question texts**
- **0 duplicate question mappings**

The dataset included direct, paraphrased, conversational, policy-focused, and unsupported questions across Easy, Medium, and Hard difficulty levels.

---

## 17.2 Baseline LLM Evaluation

A baseline LLM evaluation was performed before introducing retrieval grounding.

Because the Gemini API free-tier quota was reached during development, only a limited sample of **5 questions** could be evaluated.

The observed sample produced:

- Average LLM score: **6.4 / 10**
- PASS rate: **60%**
- FAIL rate: **40%**

These results are treated as a **development sample rather than a statistically representative benchmark**.

The baseline evaluation demonstrated that an LLM can produce fluent answers that are incomplete or broader than the approved company policy.

This provided the motivation for introducing Retrieval-Augmented Generation (RAG).

---

## 17.3 Semantic Retrieval Performance

The initial semantic retrieval system achieved:

- **84.30% retrieval accuracy** across supported questions
- **0.6232 average similarity**
- **15.70% retrieval error rate**

The evaluation showed that incorrect retrievals were concentrated around policies with similar meanings.

Examples included:

- Invoice requests vs invoice information
- Order tracking vs international delivery
- Warranty period vs warranty claims
- Order status vs order cancellation
- Payment methods vs refund payment methods

This demonstrated that semantic similarity alone can struggle to distinguish closely related policies.

---

## 17.4 Retrieval Threshold Optimisation

A threshold analysis was performed to identify a practical operating point.

The final similarity threshold was set to:

**0.50**

At this threshold:

- **88.82%** of supported questions were accepted.
- **88.86%** of accepted supported questions retrieved the correct policy.
- **100%** of unsupported challenge questions were rejected.

The threshold therefore provided a practical balance between:

- retrieval coverage
- retrieval accuracy
- unsupported-query rejection

Lower thresholds increased coverage but allowed more unsupported questions through.

Higher thresholds improved precision but rejected a larger proportion of legitimate supported questions.

---

## 17.5 Final Retrieval Failure Analysis

The final evaluation identified **98 supported-question failures**.

These were divided into:

### Threshold Rejections

**52 questions**

The system identified a potentially relevant policy, but its similarity score was below the `0.50` acceptance threshold.

These failures indicate potential opportunities for:

- better policy descriptions
- query rewriting
- improved embeddings
- category-aware retrieval

### Wrong Accepted Retrievals

**46 questions**

The retrieved policy exceeded the threshold but was still incorrect.

These represent genuine semantic confusion and are more difficult to solve simply by increasing the threshold.

The largest policy confusion patterns included:

- `INV001 → INV002`: **15 cases**
- `SHP003 → INT002`: **8 cases**
- `WAR001 → WAR002`: **8 cases**
- `ORD003 → ORD001`: **4 cases**
- `PAY001 → REF002`: **4 cases**

This indicates that retrieval errors were systematic rather than random.

---

## 17.6 Category-Level Findings

Retrieval performance varied substantially between policy categories.

### Strong categories

The following categories achieved 100% accuracy when an accepted retrieval was made:

- Account
- Discounts
- Subscriptions
- International Orders
- Products
- Returns

However, Products and Returns had lower coverage, meaning the main problem in those categories was often **confidence rather than incorrect accepted retrieval**.

### Weak categories

The weakest accepted-retrieval accuracy was observed in:

- **Invoices: 50.00%**
- **Warranty: 72.41%**
- **Shipping: 78.38%**
- **Payments: 85.19%**
- **Orders: 85.71%**

Invoices were particularly problematic because incorrect policies were sometimes retrieved with relatively high similarity scores.

This suggests that increasing the global threshold alone would not solve all retrieval problems.

---

## 17.7 Difficulty Analysis

Retrieval performance remained relatively stable across question difficulty levels.

| Difficulty | Coverage | Accuracy When Accepted |
|---|---:|---:|
| Easy | 87.10% | 89.63% |
| Medium | 89.78% | 88.62% |
| Hard | 89.52% | 88.29% |

The relatively small difference between difficulty levels suggests that retrieval quality was influenced more by **policy overlap and wording** than by the assigned difficulty level.

---

## 17.8 RAG Grounding

The RAG architecture successfully demonstrated grounded answer generation using retrieved company policy information.

A controlled live example retrieved the relevant policy and generated an answer based specifically on the retrieved content.

The system prompt instructed the LLM to:

- use only the retrieved policy
- avoid unsupported assumptions
- avoid using general knowledge
- state when the available policy was insufficient

Further live generation was limited by the Gemini API quota.

Therefore, the project does not claim full-scale LLM answer-quality or faithfulness evaluation across all 475 questions.

---

## 17.9 Hallucination Detection

A lightweight lexical-overlap heuristic was implemented as a baseline hallucination detector.

The detector classifies answers as:

- `LOW`
- `HIGH`
- `UNKNOWN`

A grounded RAG answer that closely matched its retrieved policy was classified as **LOW hallucination risk**.

This detector is intentionally treated as a baseline rather than a definitive hallucination detector.

A production implementation should combine:

- semantic similarity
- claim-level verification
- LLM-as-a-judge
- human evaluation
- contradiction detection

---

## 17.10 Automated Testing

The core retrieval and hallucination functions were tested using `pytest`.

The test suite contained:

- **4 tests**
- **4 passed**
- **0 failed**

The tests verified:

1. Policy retrieval
2. Similarity score validity
3. Low hallucination risk for grounded answers
4. High hallucination risk for unsupported answers

The automated tests provide regression protection for the core functions.

They do not replace the larger 475-question evaluation dataset, which measures real retrieval behaviour.

---

## 17.11 Overall Findings

The QualityGuard platform demonstrates that LLM evaluation should not rely solely on whether an answer sounds convincing.

The evaluation identified three important layers of AI quality:

### 1. Retrieval Quality

Can the system find the correct trusted information?

### 2. Answer Quality

Does the LLM produce a correct and relevant answer?

### 3. Grounding

Is the answer actually supported by the retrieved information?

The project therefore moves beyond simple LLM prompting toward a measurable **AI quality and data evaluation workflow**.

The current system provides a reproducible baseline that can be improved through hybrid retrieval, reranking, metadata-aware search, stronger evaluation datasets, and production-grade monitoring.

# 18. Limitations

Although QualityGuard provides a measurable evaluation framework, several limitations should be considered when interpreting the results.

---

## 18.1 LLM API Quota

The Gemini API free-tier request quota was reached during development.

As a result:

- only a limited sample of baseline LLM answers could be evaluated
- only one live RAG generation was successfully demonstrated
- full-scale answer-quality evaluation across all 475 questions was not possible
- comprehensive faithfulness and hallucination evaluation could not be completed using live LLM calls

The retrieval evaluation was therefore developed and tested independently of the LLM generation stage.

This prevents the project from overstating the current answer-quality results.

---

## 18.2 Evaluation Dataset

The evaluation dataset was programmatically generated from the company policy knowledge base.

Although it contains multiple question styles and difficulty levels, it is not equivalent to a large collection of real customer conversations.

A production evaluation dataset should include:

- real anonymised customer questions
- human-written paraphrases
- ambiguous questions
- adversarial questions
- edge cases
- multilingual queries
- previously observed failure cases

Human-authored evaluation data would provide a stronger benchmark.

---

## 18.3 Semantic Retrieval Limitations

The retrieval system uses sentence embeddings and cosine similarity.

This provides a strong baseline but can struggle when policies have very similar meanings.

Examples observed during evaluation include:

- invoice information vs invoice requests
- warranty period vs warranty claims
- order status vs order cancellation
- order tracking vs international delivery

These errors demonstrate that semantic similarity alone is not always sufficient for policy-level retrieval.

Potential improvements include:

- hybrid keyword and semantic search
- metadata filtering
- query rewriting
- cross-encoder reranking
- policy-specific metadata
- improved policy descriptions

---

## 18.4 Global Similarity Threshold

A global similarity threshold of `0.50` was selected based on the evaluation dataset.

The threshold achieved:

- 88.82% supported-question coverage
- 88.86% accuracy when accepted
- 100% unsupported-query rejection

However, different policy categories may benefit from different thresholds.

For example, Returns had relatively low coverage while Invoices experienced significant semantic confusion.

A production system could therefore use category-specific thresholds or a learned confidence model.

---

## 18.5 Hallucination Detection

The current hallucination detector uses lexical overlap between the generated answer and retrieved policy.

This is intentionally a simple baseline.

Lexical overlap cannot reliably determine whether every factual claim is correct.

For example, an answer could contain many words from the policy while still introducing an incorrect claim.

A stronger production approach would combine:

- claim extraction
- semantic entailment
- contradiction detection
- LLM-as-a-judge
- structured evaluation criteria
- human review

The current detector should therefore be interpreted as a **hallucination risk indicator**, not a definitive hallucination classifier.

---

## 18.6 LLM-as-a-Judge Limitations

The baseline evaluation uses an LLM to evaluate another LLM's answer.

This introduces potential evaluator bias.

An LLM judge may:

- favour fluent answers
- overlook subtle factual errors
- interpret ambiguous answers inconsistently
- produce inconsistent scores

A production evaluation framework should use structured scoring criteria and ideally compare LLM evaluation against human-labelled examples.

---

## 18.7 Limited RAG Evaluation

The RAG architecture was successfully demonstrated, but the live generation stage could not be evaluated across the full dataset because of API quota limitations.

Therefore, the project currently provides stronger evidence for:

**retrieval quality**

than for:

**full end-to-end generated-answer quality**.

This distinction is important when interpreting the project results.

---

## 18.8 Prototype Database

DuckDB is used for local analytical SQL processing.

This is appropriate for the prototype and evaluation workflow, but a production deployment could use PostgreSQL for:

- concurrent access
- persistent application storage
- larger workloads
- production integrations
- database permissions and governance

---

## 18.9 Airflow and Docker Deployment

Airflow and Docker artifacts were created to demonstrate workflow orchestration and reproducible deployment structure.

The current notebook environment does not represent a full production deployment of:

- Airflow
- PostgreSQL
- Power BI
- the LLM API

A production implementation would deploy these components as persistent services rather than notebook-managed components.

---

## 18.10 Power BI Reporting

The Power BI layer currently uses exported analytical datasets prepared by Python and SQL.

Automated production refresh and live monitoring were not implemented in the notebook environment.

A future implementation could connect Power BI directly to the production database and schedule automated refreshes.

---

## Overall Limitation

The main limitation of the current project is therefore not the retrieval methodology itself, but the fact that the **full end-to-end LLM evaluation stage could not be executed at scale because of API availability and quota constraints**.

The project addresses this by clearly separating:

- demonstrated results
- development samples
- retrieval benchmarks
- prototype components
- future production improvements

This ensures that the reported performance metrics remain reproducible and appropriately scoped.

# 19. Future Improvements

QualityGuard provides a working baseline for evaluating LLM responses and retrieval quality.

The following improvements would strengthen the platform for production use.

---

## 19.1 Hybrid Retrieval

The current system relies primarily on semantic embeddings.

A production system could combine:

- semantic vector search
- keyword search
- metadata filtering

This would help distinguish policies containing similar concepts but different business rules.

For example, invoice-related queries could be filtered using policy metadata before semantic ranking is applied.

---

## 19.2 Cross-Encoder Reranking

A cross-encoder could be introduced after initial vector retrieval.

The pipeline could become:

Question
→ Vector Retrieval
→ Top-K Candidates
→ Cross-Encoder Reranking
→ Best Policy
→ LLM

This would allow the system to compare the question and candidate policies more directly and potentially reduce high-confidence semantic confusion.

---

## 19.3 Query Rewriting

Customer questions could first be rewritten into a clearer retrieval query.

For example:

Customer question
→ Query understanding
→ Retrieval-optimised query
→ Semantic search

This could improve retrieval for conversational or vague questions.

---

## 19.4 Metadata-Aware Retrieval

Each policy could include structured metadata such as:

- category
- policy type
- product area
- customer intent
- policy priority
- effective date
- region

Retrieval could then combine semantic similarity with business metadata.

This would be particularly useful for separating closely related policies.

---

## 19.5 Larger Human-Authored Evaluation Dataset

The current dataset is useful for benchmarking the prototype but should be expanded with human-authored examples.

A production evaluation set could include:

- real anonymised customer queries
- expert-written paraphrases
- ambiguous queries
- adversarial queries
- edge cases
- multilingual queries
- historical failure cases

Each example could receive a human-verified expected policy and expected answer.

---

## 19.6 Stronger Hallucination Detection

The current lexical-overlap detector provides a simple baseline.

A stronger system could perform claim-level verification:

Generated answer
→ Extract factual claims
→ Compare claims against retrieved policy
→ Detect unsupported claims
→ Calculate faithfulness score

This would allow the system to distinguish between:

- supported claims
- partially supported claims
- unsupported claims
- contradictory claims

---

## 19.7 Structured LLM-as-a-Judge

The current LLM evaluation approach can be improved using structured outputs.

Instead of parsing free-form text such as:

`Score: 8`

the evaluator could return structured fields such as:

```text
{
    "correctness": 8,
    "relevance": 9,
    "grounding": 7,
    "completeness": 8,
    "overall_score": 8,
    "reason": "..."
}

## 19.8 Production PostgreSQL Database

DuckDB is appropriate for the analytical prototype.

A production implementation could migrate persistent evaluation results to PostgreSQL.

The database could store:

- evaluation results
- model versions
- retrieval results
- similarity scores
- evaluation scores
- hallucination results
- timestamps
- dataset versions

This would enable historical monitoring and trend analysis.

---

## 19.9 Production Airflow Pipeline

The current Airflow DAG demonstrates the orchestration structure.

A production implementation could schedule the complete workflow:

Dataset ingestion  
→ Retrieval  
→ RAG generation  
→ LLM evaluation  
→ Hallucination detection  
→ SQL storage  
→ Dashboard refresh

Airflow could also provide:

- retry handling
- task monitoring
- logging
- failure alerts
- scheduled evaluation runs

---

## 19.10 Dockerised Production Environment

The current Docker configuration provides a reproducible application environment.

A more complete deployment could containerise:

- QualityGuard application
- PostgreSQL
- Airflow
- supporting services

Docker Compose could then provide a reproducible local development environment.

---

## 19.11 Automated Power BI Monitoring

The current Power BI layer uses prepared analytical datasets.

A production implementation could connect Power BI directly to the database.

The dashboard could then monitor:

- retrieval accuracy
- hallucination rate
- average LLM score
- unsupported-query rate
- policy-level failures
- model performance over time
- evaluation drift

---

## 19.12 Model Comparison

QualityGuard could evaluate multiple LLMs using the same evaluation dataset.

For example:

**Model A vs Model B vs Model C**

The evaluation framework could compare:

- correctness
- relevance
- faithfulness
- hallucination rate
- latency
- cost
- retrieval dependency

This would turn QualityGuard into a broader **LLM benchmarking platform**.

---

## 19.13 CI/CD and Automated Regression Testing

GitHub Actions could automatically run the test suite whenever code changes are pushed.

A CI pipeline could perform:

1. Unit tests
2. Retrieval regression tests
3. Data validation
4. Code quality checks
5. Docker build validation

This would help prevent changes to the retrieval system from silently reducing performance.

---

## 19.14 Monitoring and Drift Detection

A production system should continuously monitor whether performance changes over time.

Potential monitoring metrics include:

- retrieval similarity distribution
- retrieval accuracy
- hallucination rate
- unsupported-query frequency
- category-level failure rate
- model response quality
- changes in customer query patterns

Sudden changes could trigger alerts for investigation.

---

# Final Vision

The long-term goal is to evolve QualityGuard from a notebook-based prototype into a production AI evaluation platform.

The proposed architecture is:

```text
Customer Query
        ↓
Query Understanding
        ↓
Hybrid Retrieval
        ↓
Reranking
        ↓
Grounded LLM Response
        ↓
Claim Verification
        ↓
LLM-as-a-Judge
        ↓
Quality Score
        ↓
PostgreSQL
        ↓
Airflow Monitoring
        ↓
Power BI Dashboard